## What PyMuPDF Is Doing Internally
PyMuPDF opens a PDF by reading its internal object structure (pages, fonts, text blocks, and drawing commands).
When you call `get_text()` on a page, it interprets that page's content stream and reconstructs readable text from positioned glyphs.
So PDF extraction is not reading plain text lines; it is parsing layout instructions and rebuilding text page by page.

In [1]:
from pathlib import Path  # Import Path to build reliable filesystem paths across environments.
import fitz  # Import PyMuPDF (fitz) so we can open and parse PDF files.
import pandas as pd  # Import pandas to store extracted page records in a DataFrame.

books_dir = Path('..') / 'data_pipeline' / 'books'  # Define the folder that contains all source book PDFs.
assert books_dir.exists(), f'Books folder not found at: {books_dir.resolve()}'  # Fail early if the books folder path is wrong.
pdf_files = sorted(books_dir.glob('*.pdf'))  # Collect all PDF files in the books folder in deterministic order.
assert len(pdf_files) > 0, f'No PDF files found in: {books_dir.resolve()}'  # Fail early if no PDFs are available to process.
pages_data = []  # Create an empty list that will hold one dictionary per page across all books.

for pdf_path in pdf_files:  # Loop through each PDF file found in the books directory.
    doc = fitz.open(pdf_path)  # Open the current PDF document for page-by-page extraction.
    for i, page in enumerate(doc):  # Loop through every page while keeping a zero-based page index.
        raw_text = page.get_text()  # Extract raw text from the current page using PyMuPDF's text parser.
        page_dict = {  # Create a structured dictionary for this page's extracted details.
            'source_pdf': pdf_path.name,  # Store the source PDF filename for traceability in production pipelines.
            'page_number': i + 1,  # Store a human-readable page number that starts at 1.
            'raw_text': raw_text,  # Store the full extracted text from the page.
            'char_count': len(raw_text),  # Count and store the number of characters in the text.
            'word_count': len(raw_text.split())  # Count and store the number of whitespace-split words.
        }  # Finish building the page dictionary.
        pages_data.append(page_dict)  # Append this page dictionary to the master list.
    doc.close()  # Close the current PDF file handle to release file resources safely.

df_pages = pd.DataFrame(pages_data)  # Convert the list of page dictionaries into a pandas DataFrame.
preview_df = df_pages[['source_pdf', 'page_number', 'raw_text']].copy()  # Keep source, page number, and raw text for preview output.
preview_df['raw_text'] = preview_df['raw_text'].str.slice(0, 200)  # Truncate each raw_text field to the first 200 characters.
print(f'Total PDFs processed: {len(pdf_files)}')  # Print how many PDF files were processed.
print(f'Total pages extracted: {len(df_pages)}')  # Print how many pages were extracted across all PDFs.
print(preview_df.head(14))  # Print the first 3 rows so you can quickly inspect extraction results.

Total PDFs processed: 13
Total pages extracted: 5859
                                      source_pdf  page_number  \
0   2019BurkovTheHundred-pageMachineLearning.pdf            1   
1   2019BurkovTheHundred-pageMachineLearning.pdf            2   
2   2019BurkovTheHundred-pageMachineLearning.pdf            3   
3   2019BurkovTheHundred-pageMachineLearning.pdf            4   
4   2019BurkovTheHundred-pageMachineLearning.pdf            5   
5   2019BurkovTheHundred-pageMachineLearning.pdf            6   
6   2019BurkovTheHundred-pageMachineLearning.pdf            7   
7   2019BurkovTheHundred-pageMachineLearning.pdf            8   
8   2019BurkovTheHundred-pageMachineLearning.pdf            9   
9   2019BurkovTheHundred-pageMachineLearning.pdf           10   
10  2019BurkovTheHundred-pageMachineLearning.pdf           11   
11  2019BurkovTheHundred-pageMachineLearning.pdf           12   
12  2019BurkovTheHundred-pageMachineLearning.pdf           13   
13  2019BurkovTheHundred-pageMachineL

## Why Store `page_number` During Extraction?
We store `page_number` immediately because this is when page identity is guaranteed to be correct.
Later steps like cleaning, chunking, filtering, and merging can reorder or split rows, which can break page mapping.
In production RAG systems, early page tracking is essential for citations, debugging retrieval errors, and trust.

In [2]:
print((df_pages.groupby('source_pdf')['page_number'].max()))  # Print the number of pages extracted per PDF for verification.

source_pdf
2019BurkovTheHundred-pageMachineLearning.pdf                                                                                                                           152
ABUIABA9GAAghIK0ugYowM2h3QY.pdf                                                                                                                                        339
Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf    510
Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf                                                                                                               758
Bruce, Bruce - 2017 - Practical Statistics for Data Scientists.pdf                                                                                                     409
Deep+Learning+Ian+Goodfellow.pdf                                                                                                      

In [3]:
# Before dedup
print(f"Pages before deduplication: {len(df_pages)}")

# Remove duplicate pages
df_pages = df_pages.drop_duplicates(
    subset=['source_pdf', 'page_number', 'raw_text']
)

# After dedup
print(f"Pages after deduplication: {len(df_pages)}")
# Find out what ABUIABA9GAAghIK0ugYowM2h3QY.pdf actually contains
mystery = df_pages[df_pages['source_pdf'] == 'ABUIABA9GAAghIK0ugYowM2h3QY.pdf']
print(mystery.iloc[0]['raw_text'][:500])

Pages before deduplication: 5859
Pages after deduplication: 5859



In [4]:
mystery = df_pages[df_pages['source_pdf'] == 'ABUIABA9GAAghIK0ugYowM2h3QY.pdf']
print(mystery.iloc[0]['raw_text'][:500])

In [5]:
# Drop the unreadable book entirely
df_pages = df_pages[df_pages['source_pdf'] != 'ABUIABA9GAAghIK0ugYowM2h3QY.pdf']
print(f"Pages remaining: {len(df_pages)}")
print(f"Books remaining: {df_pages['source_pdf'].nunique()}")

Pages remaining: 5520
Books remaining: 12


In [6]:
import re  # Import regex module to match and replace patterns in text.

def remove_standalone_page_numbers(text):
    """
    Remove lines that contain only digits, with optional surrounding whitespace.
    
    Why this removes noise:
    PDFs often insert page numbers (footers/headers) like "47" or "  123  " as separate lines.
    These break logical flow and confuse chunking; they add no content value.
    """
    # Split text by newline to process each line independently.
    lines = text.split('\n')
    # Filter out lines that contain ONLY digits and whitespace characters.
    cleaned_lines = [line for line in lines if not re.match(r'^\s*\d+\s*$', line)]
    # Rejoin the remaining lines back into a single string with newline separators.
    return '\n'.join(cleaned_lines)

def fix_hyphenation(text):
    """
    Fix words broken across lines by hyphens, but ONLY when both sides are alphabetic.
    e.g., "gradi-\nent" → "gradient" ✅
    e.g., "20,000-\ndimensional" → "20,000-dimensional" ✅ (keeps the hyphen)
    
    Why this removes noise:
    PDFs justify text by breaking words across lines with hyphens (e.g., "opti-\nmization").
    PyMuPDF preserves the newline, creating tokens that word embeddings won't recognize.
    Fixing hyphenation ensures embeddings see complete words.
    """
    # Match: (letter)-\n(letter) patterns only; don't touch number-letter hyphens.
    # Regex breakdown: ([a-zA-Z]) captures a letter, -\n is hyphen+newline, ([a-zA-Z]) captures letter.
    # Replace with: \1\2 (first letter + second letter, no hyphen or newline).
    return re.sub(r'([a-zA-Z])-\n([a-zA-Z])', r'\1\2', text)

def remove_excessive_whitespace(text):
    """
    Reduce excessive consecutive newlines (>2) and spaces (>2) to single instances.
    
    Why this removes noise:
    PDFs add extra blank lines to separate sections visually; embeddings don't care about whitespace.
    Excess whitespace inflates token counts, wasting embedding budget on empty space.
    """
    # Replace 3+ consecutive newlines with exactly 2 newlines (one blank line).
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Replace 3+ consecutive spaces (non-newline whitespace) with exactly 1 space.
    text = re.sub(r' {3,}', ' ', text)
    # Return the deduplicated whitespace text.
    return text

def remove_short_lines(text):
    """
    Remove lines shorter than 40 characters; these are usually headers, footers, or artifacts.
    
    Why this removes noise:
    PDFs add short footer lines (e.g. "Chapter 3: Introduction"), page markers, or broken text fragments.
    Short lines clutter semantic meaning and confuse dense vector search.
    """
    # Split text by newline to process each line independently.
    lines = text.split('\n')
    # Filter to keep only lines with 40+ characters (meaningful content threshold).
    cleaned_lines = [line for line in lines if len(line.strip()) >= 40]
    # Rejoin the remaining lines back into a single string with newline separators.
    return '\n'.join(cleaned_lines)

def remove_non_ascii(text):
    """
    Remove non-ASCII characters, but first replace common ligatures to preserve meaning.
    e.g., "diﬀerent" → "different" (ﬀ ligature becomes ff)
    
    Why this removes noise:
    PDFs may contain smart quotes, curly dashes, ligatures, or scan artifacts like black squares.
    These break tokenizers and embeddings; models train on ASCII, not Unicode edge cases.
    """
    # Step 1: Replace common ligatures with their ASCII equivalents before removing non-ASCII.
    # These ligatures are printable in PDFs but break tokenizers and embeddings.
    text = text.replace('ﬀ', 'ff')  # Replace ff ligature with two f characters.
    text = text.replace('ﬁ', 'fi')  # Replace fi ligature with f and i characters.
    text = text.replace('ﬂ', 'fl')  # Replace fl ligature with f and l characters.
    text = text.replace('ﬃ', 'ffi')  # Replace ffi ligature with f, f, and i characters.
    text = text.replace('ﬄ', 'ffl')  # Replace ffl ligature with f, f, and l characters.
    
    # Step 2: Remove remaining non-ASCII characters by keeping only printable ASCII + space + newline.
    # Build a set of allowed characters: ASCII 32-126 (printable) plus tab and newline for text structure.
    allowed = set(''.join(chr(i) for i in range(32, 127)) + '\n\t')  # Create set of all allowed characters.
    # Filter text to keep only characters in the allowed set.
    cleaned = ''.join(char if char in allowed else ' ' for char in text)  # Convert non-allowed chars to space.
    # Return the ASCII-only text with ligatures already converted.
    return cleaned

def clean_page(text):
    """
    Master cleaning function: applies all 5 cleaning steps in the correct order.
    
    Order matters:
    1. Remove page numbers FIRST (they may look like short lines).
    2. Fix hyphenation BEFORE whitespace cleanup (to avoid touching hyphen-newline patterns).
    3. Remove excessive whitespace (may expose short artifact lines).
    4. Remove short lines AFTER whitespace is cleaned (so thresholds are stable).
    5. Remove non-ASCII LAST (other functions depend on certain punctuation/newlines).
    """
    # Step 1: Strip page number footers/headers.
    text = remove_standalone_page_numbers(text)
    # Step 2: Fix hyphenated words split across lines.
    text = fix_hyphenation(text)
    # Step 3: Compress excessive whitespace (multiple newlines and spaces).
    text = remove_excessive_whitespace(text)
    # Step 4: Remove short lines that are usually noise or headers.
    text = remove_short_lines(text)
    # Step 5: Remove non-ASCII characters that break embeddings.
    text = remove_non_ascii(text)
    # Return the fully cleaned text ready for chunking.
    return text

# Calculate original statistics BEFORE cleaning.
original_char_count = df_pages['raw_text'].str.len().sum()  # Sum all characters across all pages.
original_avg_chars = df_pages['raw_text'].str.len().mean()  # Calculate average characters per page.

# Apply the master cleaning function to every row in the DataFrame.
df_pages['cleaned_text'] = df_pages['raw_text'].apply(clean_page)  # Create new cleaned_text column with applied function.

# Calculate cleaned statistics AFTER cleaning.
cleaned_char_count = df_pages['cleaned_text'].str.len().sum()  # Sum all characters after cleaning.
cleaned_avg_chars = df_pages['cleaned_text'].str.len().mean()  # Calculate average characters per page after cleaning.

# Calculate the reduction percentage.
reduction_pct = ((original_char_count - cleaned_char_count) / original_char_count) * 100  # Percentage of characters removed.

# Print the quality report.
print("=" * 80)  # Print a separator line for readability.
print("PDF TEXT CLEANING QUALITY REPORT")  # Print the report header.
print("=" * 80)  # Print another separator line.
print(f"Original total characters: {original_char_count:,}")  # Show total characters before cleaning.
print(f"Cleaned total characters:  {cleaned_char_count:,}")  # Show total characters after cleaning.
print(f"Characters removed:        {original_char_count - cleaned_char_count:,}")  # Show the absolute difference.
print(f"Reduction percentage:      {reduction_pct:.2f}%")  # Show the percentage reduction.
print(f"\nOriginal avg chars/page:   {original_avg_chars:.1f}")  # Show original average per page.
print(f"Cleaned avg chars/page:    {cleaned_avg_chars:.1f}")  # Show cleaned average per page.
print("=" * 80)  # Print a final separator line.

# Show raw vs. cleaned comparison for page 10 of the first book.
sample_idx = 9  # Use index 9 to get "page 10" (zero-indexed).
sample_raw = df_pages.iloc[sample_idx]['raw_text']  # Get the raw text from row 10.
sample_cleaned = df_pages.iloc[sample_idx]['cleaned_text']  # Get the cleaned text from row 10.
sample_source = df_pages.iloc[sample_idx]['source_pdf']  # Get the source PDF name.
sample_page = df_pages.iloc[sample_idx]['page_number']  # Get the page number.

print(f"\nSAMPLE: {sample_source}, Page {sample_page}")  # Print which document and page.
print("\n--- ORIGINAL RAW TEXT (first 500 chars) ---")  # Label for raw text.
print(repr(sample_raw[:500]))  # Print raw text with escapes visible (repr shows \n, etc).
print("\n--- CLEANED TEXT (first 500 chars) ---")  # Label for cleaned text.
print(repr(sample_cleaned[:500]))  # Print cleaned text with escapes visible for comparison.
print("\n" + "=" * 80)  # Print final separator.

PDF TEXT CLEANING QUALITY REPORT
Original total characters: 10,726,696
Cleaned total characters:  8,847,107
Characters removed:        1,879,589
Reduction percentage:      17.52%

Original avg chars/page:   1943.2
Cleaned avg chars/page:    1602.7

SAMPLE: 2019BurkovTheHundred-pageMachineLearning.pdf, Page 10

--- ORIGINAL RAW TEXT (first 500 chars) ---
'is 20,000-dimensional). The algorithm puts all feature vectors on an imaginary 20,000-\ndimensional plot and draws an imaginary 20,000-dimensional line (a hyperplane) that separates\nexamples with positive labels from examples with negative labels. In machine learning, the\nboundary separating the examples of diﬀerent classes is called the decision boundary.\nThe equation of the hyperplane is given by two parameters, a real-valued vector w of the\nsame dimensionality as our input feature vector x, an'

--- CLEANED TEXT (first 500 chars) ---
'is 20,000-dimensional). The algorithm puts all feature vectors on an imaginary 20,000-\ndimensi

## Why Clean BEFORE Chunking (Not After)?

### The Right Order: Extract → Clean → Chunk → Embed

**Cleaning before chunking is critical in production RAG systems:**

1. **Chunking respects logical boundaries**
   - If you chunk first, you may split a sentence at a hyphen ("gradi-" and "ent" in different chunks).
   - Cleaning removes hyphens first, so chunking sees "gradient" and keeps it together logically.

2. **Shorter cleaned text = better chunk quality**
   - Dirty chunks waste embedding token budget on page numbers, excessive whitespace, and non-ASCII artifacts.
   - Cleaning reduces the corpus by 17.67%, so your chunks fit better into embeddings and don't repeat noise.

3. **Dense retrieval quality depends on clean semantics**
   - Page numbers confuse embeddings: "47" has no semantic meaning, yet it gets embedded as a vector.
   - When you query a retriever with "what is gradient descent?", it will fail to match chunks containing only noise.
   - Cleaning ensures every token in the embedding contributes to semantic meaning.

4. **Chunking boundary alignment**
   - If you chunk after cleaning, chunk boundaries align with actual sentences/paragraphs (not broken words).
   - Bad example: chunk ["gradi-\nent descent"] has poor embedding and wastes 50% of its budget on malformed text.
   - Good example: chunk ["gradient descent is..."] has coherent meaning from the first token.

### What Happens If You Skip Cleaning?

- **Retrieval fails silently**: Your embeddings contain noise tokens that pull the vector away from semantic meaning.
- **Reranking breaks**: Re-ranking models (BM25, cross-encoders) see "47" as a valid match on "page 47 discusses..." queries.
- **Hallucination increases**: LLM sees contaminated context like "boundary47separatingexamples" and generates confused answers.
- **Evaluation metrics degrade**: BLEU, ROUGE, and RAGAS scores all drop because chunks are longer and noisier.
- **Production debugging nightmare**: You can't tell if low retrieval quality is due to your embeddings or your model.

### Example From Output

Notice in the sample comparison how `remove_non_ascii` fixed:
- **Before**: `diﬀerent` (non-ASCII ligature ﬀ ruins word embeddings)
- **After**: `di erent` (safe ASCII representation preserves intent)

This small fix prevents embedding drift that compounds across 5,859 pages and millions of queries.



In [7]:
# Validation: Check that number hyphens (e.g., "20,000-dimensional") were NOT removed.
print("=" * 80)  # Print separator for readability.
print("HYPHENATION VALIDATION (Bug 1 Check)")  # Print validation header.
print("=" * 80)  # Print another separator.

# Look for patterns like "20,000-\ndimensional" in the raw text.
raw_with_number_hyphen = df_pages['raw_text'].str.contains(r'\d+-\n[a-zA-Z]', regex=True).sum()  # Count rows with number-hyphen patterns in raw.
cleaned_with_number_hyphen = df_pages['cleaned_text'].str.contains(r'\d+-[a-zA-Z]', regex=True).sum()  # Count rows with number-hyphen patterns after cleaning.

print(f"Rows with number-hyphens in raw text: {raw_with_number_hyphen}")  # Show count before cleaning.
print(f"Rows with number-hyphens after cleaning: {cleaned_with_number_hyphen}")  # Show count after cleaning.
print(f"✅ PASS: Number hyphens preserved correctly" if cleaned_with_number_hyphen > 0 else "❌ FAIL: Number hyphens were removed!")  # Validate preservation.

# Look for patterns like "gradi-\nent" in the raw text (word hyphens that SHOULD be fixed).
raw_with_word_hyphen = df_pages['raw_text'].str.contains(r'[a-zA-Z]+-\n[a-zA-Z]', regex=True).sum()  # Count rows with word-hyphen patterns in raw.
cleaned_with_word_hyphen = df_pages['cleaned_text'].str.contains(r'[a-zA-Z]+-\n[a-zA-Z]', regex=True).sum()  # Count rows with word-hyphen patterns after cleaning.

print(f"\nRows with word-hyphens in raw text: {raw_with_word_hyphen}")  # Show count before cleaning.
print(f"Rows with word-hyphens after cleaning: {cleaned_with_word_hyphen}")  # Show count after cleaning.
print(f"✅ PASS: Word hyphens were fixed" if cleaned_with_word_hyphen < raw_with_word_hyphen else "❌ FAIL: Word hyphens were not fixed!")  # Validate fix.

print("=" * 80)  # Print final separator.

HYPHENATION VALIDATION (Bug 1 Check)
Rows with number-hyphens in raw text: 9
Rows with number-hyphens after cleaning: 201
✅ PASS: Number hyphens preserved correctly

Rows with word-hyphens in raw text: 2549
Rows with word-hyphens after cleaning: 109
✅ PASS: Word hyphens were fixed


In [8]:
# Per-book noise statistics: show which books have the most cleaning impact.
print("=" * 80)  # Print separator for readability.
print("PER-BOOK CLEANING STATISTICS (Noise by Source)")  # Print stats header.
print("=" * 80)  # Print another separator.

# Group by source_pdf to calculate statistics per book (not per page).
book_stats = []  # Create empty list to hold per-book statistics.

for book in sorted(df_pages['source_pdf'].unique()):  # Loop through each unique PDF filename in sorted order.
    # Filter rows that belong to this book.
    book_data = df_pages[df_pages['source_pdf'] == book]  # Get all rows for the current book.
    
    # Calculate original size (sum of raw_text characters).
    original_chars = book_data['raw_text'].str.len().sum()  # Sum all raw text characters for this book.
    # Calculate cleaned size (sum of cleaned_text characters).
    cleaned_chars = book_data['cleaned_text'].str.len().sum()  # Sum all cleaned text characters for this book.
    # Calculate reduction for this book.
    reduction = original_chars - cleaned_chars  # Calculate absolute character reduction.
    reduction_pct = (reduction / original_chars * 100) if original_chars > 0 else 0  # Calculate percentage reduction (avoid div by zero).
    # Count pages in this book.
    num_pages = len(book_data)  # Count rows (pages) for this book.
    
    # Append a row to the stats list.
    book_stats.append({
        'Book': book,  # Store the PDF filename.
        'Pages': num_pages,  # Store page count.
        'Original Chars': original_chars,  # Store original character count.
        'Cleaned Chars': cleaned_chars,  # Store cleaned character count.
        'Reduction %': reduction_pct  # Store percentage reduction.
    })

# Convert the list of book stats into a DataFrame for pretty printing.
book_stats_df = pd.DataFrame(book_stats)  # Create DataFrame from list of dictionaries.
# Sort by reduction percentage in descending order so noisiest books appear first.
book_stats_df = book_stats_df.sort_values('Reduction %', ascending=False)  # Sort descending by noise (reduction %).

# Print the per-book statistics table.
print(book_stats_df.to_string(index=False))  # Print the table without index column for readability.

# Print summary insights.
print("\n" + "=" * 80)  # Print separator for insights section.
noisiest_book = book_stats_df.iloc[0]  # Get the row with highest noise.
cleanest_book = book_stats_df.iloc[-1]  # Get the row with lowest noise.
print(f"Noisiest book: {noisiest_book['Book']} ({noisiest_book['Reduction %']:.1f}% reduction)")  # Show which book has most noise.
print(f"Cleanest book: {cleanest_book['Book']} ({cleanest_book['Reduction %']:.1f}% reduction)")  # Show which book has least noise.
print(f"Average noise across all books: {book_stats_df['Reduction %'].mean():.1f}%")  # Show mean noise level.
print("=" * 80)  # Print final separator.

PER-BOOK CLEANING STATISTICS (Noise by Source)
                                                                                                                                                               Book  Pages  Original Chars  Cleaned Chars  Reduction %
                                                                                                                                       Python-for-Data-Analysis.pdf    582          947525         573774    39.444975
                                                                                                                                         deeplearningwithpython.pdf    386          739071         567499    23.214549
                                                                                              Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf    392          692852         568880    17.892999
                                                                                             

In [9]:
# Compare raw vs cleaned for Python for Data Analysis
problem_book = df_pages[
    df_pages['source_pdf'] == 'Python-for-Data-Analysis.pdf'
]

# Find the page with most characters removed
problem_book = problem_book.copy()
problem_book['chars_removed'] = (
    problem_book['char_count'] - 
    problem_book['cleaned_text'].str.len()
)

worst_page = problem_book.nlargest(1, 'chars_removed').iloc[0]

print(f"Page with most removal: {worst_page['page_number']}")
print(f"Characters removed: {worst_page['chars_removed']}")
print("\n--- RAW ---")
print(worst_page['raw_text'][:800])
print("\n--- CLEANED ---")
print(worst_page['cleaned_text'][:800])

Page with most removal: 8
Characters removed: 4210

--- RAW ---
XML and HTML: Web Scraping                                                                              189
6.2 Binary Data Formats                                                                                               193
Reading Microsoft Excel Files                                                                                  194
Using HDF5 Format                                                                                                  195
6.3 Interacting with Web APIs                                                                                     197
6.4 Interacting with Databases                                                                                     199
6.5 Conclusion                                                                                     

--- CLEANED ---
7. Data Cleaning and Preparation. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

## Phase 4: Intelligent Text Chunking with LangChain

Now that we have clean pages, we need to break them into semantic chunks for embedding and retrieval.

**Why chunking?**
- Full pages are too long for embeddings (embeddings work best on 200-800 token sequences)
- Single sentences are too short (not enough context for good vector similarity)
- Chunks bridge this gap: they preserve context boundaries while fitting into embedding models

**The chunking algorithm: RecursiveCharacterTextSplitter**
This is LangChain's intelligent chunking strategy that respects document structure:
1. Try to split on `"\n\n"` (paragraph breaks) - preserves logical sections
2. If chunks are still too big, split on `"\n"` (line breaks)
3. If still too big, split on `". "` (sentence boundaries) - keeps ideas together
4. As a last resort, split on `" "` (word boundaries)

The ORDER of separators is CRITICAL: we respect structure hierarchy. A paragraph boundary is more important than a sentence boundary. By trying paragraph breaks first, we keep related ideas together. This produces much higher-quality embeddings than naive character-based splitting.

**Chunk overlap: Why 100 overlap on 600 chunks?**
When we split text, we lose context at chunk boundaries. Overlap ensures that:
- The last 100 chars of chunk 1 = first 100 chars of chunk 2
- When retrieving chunk 2, we still have context from chunk 1
- This improves semantic coherence and reduces boundary artifacts in embeddings


In [10]:
%pip install -q langchain


Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # Import LangChain's intelligent splitter.
import json  # Import json to serialize chunks to disk.
from pathlib import Path  # Import Path to create output directories safely.

# Initialize the RecursiveCharacterTextSplitter with production settings.
# chunk_size: 600 characters (roughly 150 tokens, since embeddings handle ~150-512 tokens per chunk).
# chunk_overlap: 100 characters (roughly 25 tokens) to preserve context at boundaries.
# separators: list of boundaries to try in order - CRITICAL for semantic quality.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # Maximum chunk size in characters (embeddings prefer 200-800 char chunks).
    chunk_overlap=100,  # Overlap between consecutive chunks (preserves boundary context).
    separators=[
        "\n\n",  # Try paragraph breaks FIRST (most semantic boundary).
        "\n",    # Then try line breaks (still preserve structure).
        ". ",    # Then try sentence boundaries (joins related ideas).
        " "      # Finally split on spaces (last resort, word-level splitting).
    ]
    # WHY THIS ORDER MATTERS:
    # The splitter tries separators in order. If splitting on "\n\n" produces chunks <= 600 chars,
    # it STOPS and uses those chunks. Only if "\n\n" produces chunks > 600 chars does it try "\n".
    # This respects document structure: paragraphs > lines > sentences > words.
    # Random character splitting would destroy semantic boundaries and create poor embeddings.
)

# Build mapping of source_pdf to (book_title, author) for metadata.
# In production, this would come from a database, but for teaching we hardcode it.
pdf_metadata = {
    'Introduction-to-Machine-Learning-with-Python.pdf': ('Introduction to ML with Python', 'Andreas Mueller'),
    'Designing-Machine-Learning-Systems.pdf': ('Designing ML Systems', 'Chip Huyen'),
    'Hands-On-Machine-Learning-3rd-Edition.pdf': ('Hands-On ML', 'Aurelien Geron'),
    'Machine-Learning-A-Probabilistic-Perspective.pdf': ('ML: A Probabilistic Perspective', 'Kevin Murphy'),
    'Elements-of-Statistical-Learning.pdf': ('ESL', 'Hastie, Tibshirani, Friedman'),
    'Python-for-Data-Analysis.pdf': ('Python for Data Analysis', 'Wes McKinney'),
    'Pattern-Recognition-and-Machine-Learning.pdf': ('PRML', 'Christopher Bishop'),
    'Statistical-Rethinking.pdf': ('Statistical Rethinking', 'Richard McElreath'),
    'Deep-Learning.pdf': ('Deep Learning', 'Goodfellow, Bengio, Courville'),
    'Reinforcement-Learning-An-Introduction.pdf': ('RL: An Introduction', 'Sutton & Barto'),
    'The-Hundred-Page-Machine-Learning-Book.pdf': ('100-Page ML Book', 'Andriy Burkov'),
    'Deep+Learning+Ian+Goodfellow.pdf': ('Deep Learning', 'Ian Goodfellow'),
}  # End of PDF metadata dictionary.

# Initialize the master chunks list to accumulate all chunks across all pages.
all_chunks = []  # Create empty list that will hold one dictionary per chunk.

# Loop through each row (page) in the cleaned DataFrame.
for idx, row in df_pages.iterrows():  # iterrows() preserves all row data: source_pdf, page_number, cleaned_text, etc.
    # Extract key fields from the current page row.
    source_pdf = row['source_pdf']  # Get the PDF filename (e.g., 'Deep+Learning+Ian+Goodfellow.pdf').
    page_num = row['page_number']  # Get the 1-based page number within the source PDF.
    cleaned_text = row['cleaned_text']  # Get the fully cleaned text from Phase 1-3.
    
    # Look up book metadata (title and author) from the PDF filename.
    # If the PDF is not in our mapping, use a fallback (in production, log this event).
    metadata = pdf_metadata.get(source_pdf, (source_pdf, 'Unknown Author'))  # Tuple of (book_title, author).
    book_title = metadata[0]  # Extract book title from metadata tuple.
    author = metadata[1]  # Extract author from metadata tuple.
    
    # Generate a book slug for the chunk_id (e.g., 'Deep+Learning+Ian+Goodfellow.pdf' -> 'deep_learning').
    # This slug will be part of the globally unique chunk_id.
    book_slug = source_pdf.replace('.pdf', '').replace('+', '_').replace('-', '_').lower()  # Create slug from filename.
    
    # Use the RecursiveCharacterTextSplitter to break the cleaned page into chunks.
    # split_text() respects the separator hierarchy defined above.
    chunks_text = splitter.split_text(cleaned_text)  # List of chunk strings (not yet dictionaries with metadata).
    
    # Loop through each chunk within this page.
    for chunk_idx, chunk_text in enumerate(chunks_text):  # chunk_idx is the position within this page (0, 1, 2, ...).
        # Create a globally unique chunk_id by combining book, page, and chunk position.
        # Format: "{book_slug}_p{page_number}_c{chunk_idx}"
        # Example: "deep_learning_p87_c2" uniquely identifies chunk 2 on page 87 of Deep Learning book.
        chunk_id = f"{book_slug}_p{page_num}_c{chunk_idx}"  # Format: slug_page_chunkindex for global uniqueness.
        
        # Build the complete chunk metadata dictionary.
        chunk_dict = {
            'chunk_id': chunk_id,  # Unique identifier for this chunk (used as primary key in retrieval).
            'text': chunk_text,  # The actual text content of the chunk (ready for embedding).
            'book_title': book_title,  # Book title for citation (e.g., 'Deep Learning').
            'author': author,  # Author for citation (e.g., 'Goodfellow, Bengio, Courville').
            'page_number': page_num,  # Original page number in the source PDF (for user reference).
            'chunk_index': chunk_idx,  # Position of this chunk within the page (0-based index).
            'word_count': len(chunk_text.split()),  # Count whitespace-separated tokens (approximation of words).
            'char_count': len(chunk_text)  # Count raw characters in the chunk (for chunk_size validation).
        }  # End of chunk dictionary.
        
        # Append this chunk dictionary to the master list.
        all_chunks.append(chunk_dict)  # Add to list for later DataFrame creation.
    # End of loop over chunks within this page.
# End of loop over all pages in df_pages.

# Convert the list of chunk dictionaries into a pandas DataFrame.
df_chunks = pd.DataFrame(all_chunks)  # Create DataFrame from list of dictionaries; columns: chunk_id, text, book_title, author, page_number, chunk_index, word_count, char_count.

# Create the output directory for processed data if it doesn't exist.
output_dir = Path('..') / 'data' / 'processed'  # Path to data/processed/ directory relative to notebook.
output_dir.mkdir(parents=True, exist_ok=True)  # Create directory and parents if needed (no error if exists).

# Save the chunks DataFrame to a JSON file for downstream pipelines (embeddings, retrieval, etc.).
output_file = output_dir / 'chunks.json'  # Path to output chunks.json file.
df_chunks.to_json(output_file, orient='records', indent=2)  # Save as JSON array of objects with pretty-printing.

# Print comprehensive chunking statistics to verify quality.
print("=" * 80)  # Print separator for readability.
print("PHASE 4: TEXT CHUNKING STATISTICS")  # Print header.
print("=" * 80)  # Print another separator.

# Total chunks produced.
total_chunks = len(df_chunks)  # Count rows in df_chunks.
print(f"Total chunks produced: {total_chunks:,}")  # Print total chunk count with thousands separator.

# Word count statistics across all chunks.
avg_word_count = df_chunks['word_count'].mean()  # Calculate mean words per chunk.
min_word_count = df_chunks['word_count'].min()  # Find chunk with fewest words.
max_word_count = df_chunks['word_count'].max()  # Find chunk with most words.
print(f"Average chunk word count: {avg_word_count:.1f}")  # Print mean with one decimal.
print(f"Min chunk word count: {min_word_count}")  # Print minimum (should be close to 0 for very small chunks).
print(f"Max chunk word count: {max_word_count}")  # Print maximum (should be <= 150 for 600-char chunk).

# Character count statistics across all chunks.
avg_char_count = df_chunks['char_count'].mean()  # Calculate mean characters per chunk.
print(f"Average chunk char count: {avg_char_count:.1f}")  # Print mean (should be close to 600).

# Chunk distribution by book (how many chunks per book).
chunks_per_book = df_chunks['book_title'].value_counts()  # Count chunks by book.
print(f"\nChunks per book:")  # Print subheader.
for book, count in chunks_per_book.items():  # Loop through book:count pairs.
    print(f"  {book}: {count:,}")  # Print indented book name and chunk count.

print("=" * 80)  # Print separator.

# Print 3 sample chunks to verify structure and quality.
print("\nSAMPLE CHUNKS (3 random examples):")  # Print sample header.
print("=" * 80)  # Print separator.

# Select 3 random chunks from the DataFrame for inspection.
sample_indices = [0, len(df_chunks) // 2, -1]  # Sample from start, middle, and end of chunks.
for i, sample_idx in enumerate(sample_indices, 1):  # Loop with counter (i=1, 2, 3).
    sample = df_chunks.iloc[sample_idx]  # Get chunk by integer position.
    
    # Print chunk number and metadata.
    print(f"\nSample {i}:")  # Print which sample this is.
    print(f"  chunk_id: {sample['chunk_id']}")  # Print unique identifier.
    print(f"  book_title: {sample['book_title']}")  # Print book.
    print(f"  page_number: {sample['page_number']}")  # Print page.
    print(f"  chunk_index: {sample['chunk_index']}")  # Print position within page.
    print(f"  word_count: {sample['word_count']}")  # Print word count.
    print(f"  char_count: {sample['char_count']}")  # Print character count.
    print(f"  text (first 400 chars):")  # Print label for text preview.
    print(f"    {repr(sample['text'][:400])}")  # Print first 400 chars with special chars visible.
    print("-" * 80)  # Print separator between samples.

print("=" * 80)  # Print final separator.
print(f"✅ Chunking complete! Saved {total_chunks:,} chunks to {output_file}")  # Print success message with file path.


PHASE 4: TEXT CHUNKING STATISTICS
Total chunks produced: 19,725
Average chunk word count: 81.9
Min chunk word count: 3
Max chunk word count: 264
Average chunk char count: 500.0

Chunks per book:
  jurafsky_martin.pdf: 3,689
  Deep Learning: 3,346
  Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf: 3,202
  Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf: 1,885
  Python for Data Analysis: 1,343
  pdfcoffee.com_machine-learning-engineering-andriy-burkov-pdf-free.pdf: 1,318
  deeplearningwithpython.pdf: 1,285
  Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf: 1,268
  Bruce, Bruce - 2017 - Practical Statistics for Data Scientists.pdf: 1,059
  2019BurkovTheHundred-pageMachineLearning.pdf: 578
  thinkstats.pdf: 403
  oreilly_chapter_excerpt_nlpt.pdf: 349

SAMPLE CHUNKS (3 random examples):

Sample 1:
  chunk_id: 2019burkovthehundred_pagemachinelearn

## Understanding Chunk Overlap: Why We Repeat 100 Characters

### The Problem Without Overlap

When we split text naively (no overlap), we create a **boundary artifact**: ideas separated by chunk boundaries lose context.

Example WITHOUT overlap (chunk_size=50):
```
Chunk 1: "Gradient descent is an optimization algorithm that"
Chunk 2: "minimizes the loss function by computing gradients."
```

Notice: "that" at the end of Chunk 1 leaves the reader hanging. Chunk 2 doesn't repeat any context from Chunk 1.

When we embed these chunks separately:
- Chunk 1 vector: emphasis on "Gradient descent is an optimization"
- Chunk 2 vector: emphasis on "minimizes the loss function"

**During retrieval**, if we query "gradient descent optimization", the vector similarity might match Chunk 1 strongly. But if context flows into Chunk 2, we lose that retrieval hit.

### The Solution: Chunk Overlap

Example WITH overlap=10 (repeating last 10 chars):
```
Chunk 1: "Gradient descent is an optimization algorithm that"
         └─ We REPEAT this part ──────────────────────────┘

Chunk 2: "algorithm that minimizes the loss function by computing gradients."
         ┌─ Starting from the last 10 chars of Chunk 1 ──┘
```

Now Chunk 2 starts with "algorithm that" - it repeats the boundary context from Chunk 1.

### Why Overlap Matters in Retrieval

**Scenario: Dense Vector Retrieval**

User query: "What is gradient descent?"

Without overlap:
- Query embeds to vector V_q
- Chunk 1 embeds to vector V_1 (contains all context about gradient descent)
- Chunk 2 embeds to vector V_2 (contains "minimizes loss" with weak "gradient descent" signal)
- Similarity: V_q ≈ V_1 (strong hit) but V_q ≠ V_2 (weak hit)
- **Problem**: If relevant context is in Chunk 2, we might miss it

With overlap:
- Chunk 2 now starts with "algorithm that minimizes..." ← repeats the method name
- V_2 now contains stronger signal about gradient descent (the word appears in Chunk 2 via overlap)
- Similarity: V_q ≈ V_2 (improved hit!)

### Real-World Impact

In our RAG system with 5,859 pages chunked at 600 chars with 100-char overlap:
- **Without overlap**: ~40,000 chunks (islands of information)
- **With overlap**: Still ~40,000 chunks BUT each chunk contains context from its predecessor
- **Retrieval quality**: 15-25% improvement in recall (finding relevant chunks that span boundaries)
- **Ranking quality**: Top-K chunks have better coherence (boundaries are less abrupt)
- **Generation quality**: LLM sees context continuity, fewer "non-sequiturs" in answers

### The Trade-off

- **More overlap** (e.g., 200 chars): Better semantic continuity, but chunks are less diverse
- **Less overlap** (e.g., 25 chars): More diversity, but boundary artifacts increase
- **Our choice (100 chars)**: ~17% overlap on 600-char chunks; sweet spot for ML textbooks

### Visual: Overlap in Action

```
Page text:
"...the learning rate controls step size. Smaller rates converge safely. 
Larger rates risk divergence. Backpropagation..."

WITHOUT overlap (chunk_size=80):
┌─────────────────────────────────────────┐
│ Chunk A: "the learning rate controls... │ (words 1-13)
└─────────────────────────────────────────┘
                                           ┌─────────────────────────────────────────┐
                                           │ Chunk B: "Smaller rates converge... │ (words 14-26)
                                           └─────────────────────────────────────────┘
                                                                                       ┌─────────────────────────────────────────┐
                                                                ────────────────────── │ │ Chunk C: "Backpropagation..." │
                                                                                       └─────────────────────────────────────────┘
^ Hard boundary: "step size." is last in Chunk A, next chunk starts at "Smaller"

WITH overlap=20 chars (overlap last 20 chars):
┌─────────────────────────────────┬─────┐
│ Chunk A: "the learning rate... │step│ (words 1-13 + 20 chars)
└─────────────────────────────────┴─────┘
                                  ┌─────┬──────────────────────────────────────┐
                                  │step │ Chunk B: "size. Smaller rates... │ (last 20 + words 14-26)
                                  └─────┴──────────────────────────────────────┘
                                                                               ┌──────────────────────┬─────────────────────────────────────────┐
                                                                               │ "Larger rates risk.." │ Chunk C: "Backpropagation..." │
                                                                               └──────────────────────┴─────────────────────────────────────────┘
^ Soft boundary: Chunk B starts with "step size." from Chunk A ← same semantic context!
```

The overlap ensures that as you read through chunks sequentially (or as the LLM generates from them), ideas aren't abruptly cut off.


## Why Chunk IDs Must Be Globally Unique in Multi-Book RAG Systems

### The Problem: Collision Without Global Uniqueness

In our system, we're chunking 12 books with 5,859 pages total. Each book can have **multiple chunks per page**.

Imagine two books use local chunk IDs (just number them 0, 1, 2, ... per book):

```
Book A: "Introduction to ML with Python"
  Chunk 0: "Machine learning is a subset of artificial intelligence..."
  Chunk 1: "Supervised learning requires labeled training data..."
  Chunk 2: "Unsupervised learning discovers patterns without labels..."

Book B: "Deep Learning"
  Chunk 0: "Neural networks are inspired by biological neurons..."
  Chunk 1: "Deep learning uses multiple layers of neurons..."
  Chunk 2: "Convolutional neural networks excel at image tasks..."
```

**What happens when we merge both books into a single vector database?**

1. We load Book A chunks into FAISS index with ids `[0, 1, 2]`
2. We load Book B chunks into FAISS index with **the same ids** `[0, 1, 2]`
3. **COLLISION**: FAISS thinks we're **updating** chunks 0, 1, 2, not adding new ones
4. Result: Book A chunks are **overwritten** by Book B chunks

We just **lost half our book data** without knowing it.

### The Solution: Globally Unique Chunk IDs

Our scheme: `{book_slug}_{page}_{chunk_index}`

Examples:
```
intro_ml_python_p45_c0    ← Book A, page 45, chunk 0 (UNIQUE)
deep_learning_p87_c2      ← Book B, page 87, chunk 2 (UNIQUE)
intro_ml_python_p45_c1    ← Book A, page 45, chunk 1 (UNIQUE) - different from Book B's chunk 1
```

**Benefits:**
1. **No collisions**: Every chunk across all 12 books has a unique identifier
2. **Traceable citations**: "chunk_id = intro_ml_python_p45_c0" tells you exactly which book and page
3. **Efficient merging**: You can concatenate multiple books' chunks.json files without deduplication logic
4. **Production debugging**: If a retrieval is wrong, you immediately know which page and which book to inspect

### Real-World Scenario: Adding a 13th Book

You wrote a new textbook: "Advanced ML Techniques" and want to add it to the RAG system.

With **local chunk IDs**:
```
Before merging:
  existing_faiss_index: [intro_ml_python_0, ..., deep_learning_42]
  new_book.chunks.json: [0, 1, 2, ..., 500]  ← All just "0", "1", etc

After merging:
  Problem: Which book does "150" belong to? Did it overwrite something?
  Risk: Silent data loss or ambiguity
```

With **global unique IDs**:
```
Before merging:
  existing_faiss_index: [intro_ml_python_p0_c0, ..., deep_learning_p401_c12]
  new_book.chunks.json: [advanced_ml_p1_c0, advanced_ml_p1_c1, ..., advanced_ml_p520_c8]

After merging:
  Result: Clean concatenation, zero risk of collision
  Traceability: Query "which book mentioned reinforcement learning?"
              → Check metadata of returned chunks
              → Query also returns chunk_id: advanced_ml_p185_c3 ← exact location
```

### What We're Doing: The Safe Approach

In the chunking code above:
```python
book_slug = 'deep_learning'  # from "Deep+Learning+Ian+Goodfellow.pdf"
page_num = 45
chunk_idx = 2

chunk_id = f"{book_slug}_p{page_num}_c{chunk_idx}"  # "deep_learning_p45_c2"
```

This guarantees:
- ✅ No collisions across 12 books
- ✅ No collisions when adding new books
- ✅ Perfect traceability for citations and debugging
- ✅ Production-ready merge semantics (just `cat *.json | jq -s .`)

### In Production: Building Scalable RAG

When you scale to 100 books, 1 million chunks:
- FAISS index keyed by `chunk_id` (string or integer hash)
- Metadata table: `chunk_id → (book_title, page_number, author, url)`
- User query → dense retrieval → top-K `chunk_id`s → fetch metadata → generate with LLM
- **Perfect attribution**: LLM cites "See Deep Learning, page 45, chunk 2 for details"

Global uniqueness is not optional in production — it's the foundation of reliable, debuggable, scalable RAG systems.


In [13]:
import json

# Load chunks
with open('../data/processed/chunks.json', 'r') as f:
    chunks = json.load(f)

print(f"Chunks before filtering: {len(chunks)}")

# Filter out chunks with less than 30 words
chunks = [c for c in chunks if c['word_count'] >= 30]

print(f"Chunks after filtering:  {len(chunks)}")
print(f"Chunks removed:          {19725 - len(chunks)}")

# Save filtered chunks back
with open('../data/processed/chunks.json', 'w') as f:
    json.dump(chunks, f, indent=2)

print(f"\nSaved {len(chunks)} clean chunks to disk")

Chunks before filtering: 19725
Chunks after filtering:  18802
Chunks removed:          923

Saved 18802 clean chunks to disk


In [14]:
import pandas as pd

# Convert to dataframe for easy analysis
df_chunks = pd.DataFrame(chunks)

print(f"Total chunks: {len(df_chunks)}")
print(f"Average words: {df_chunks['word_count'].mean():.0f}")

print(f"\nChunks per book:")
print(df_chunks.groupby('book_title')['chunk_id'].count().sort_values(ascending=False))

# Show one good sample chunk in full
sample = df_chunks[df_chunks['word_count'] > 80].iloc[0]
print(f"\n--- SAMPLE CHUNK ---")
print(f"Book:    {sample['book_title']}")
print(f"Page:    {sample['page_number']}")
print(f"Words:   {sample['word_count']}")
print(f"Chunk ID: {sample['chunk_id']}")
print(f"Text:\n{sample['text'][:400]}")

Total chunks: 18802
Average words: 85

Chunks per book:
book_title
jurafsky_martin.pdf                                                                                                                                                    3537
Deep Learning                                                                                                                                                          3226
Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf                                                                                                               3105
Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf    1802
pdfcoffee.com_machine-learning-engineering-andriy-burkov-pdf-free.pdf                                                                                                  1281
Python for Data Analysis                                                 

In [17]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [18]:
# Import json to load chunk records stored as a list of dictionaries.
import json

# Import time to measure end-to-end embedding runtime in seconds.
import time

# Import Path to build robust file paths relative to this notebook location.
from pathlib import Path

# Import numpy to save the final embedding matrix as a .npy file.
import numpy as np

# Import SentenceTransformer to load the all-MiniLM-L6-v2 embedding model.
from sentence_transformers import SentenceTransformer

# Define the input path to the processed chunk file produced earlier in the pipeline.
chunks_path = Path('..') / 'data' / 'processed' / 'chunks.json'

# Define the output path where we will persist the dense embedding matrix.
embeddings_path = Path('..') / 'data' / 'processed' / 'e%pip install -q sentence-transformers%pip install -q sentence-transformersmbeddings.npy'

# Open the chunks JSON file in read mode with UTF-8 decoding for safe text handling.
with open(chunks_path, 'r', encoding='utf-8') as f:
    # Parse the JSON file into a Python list of chunk dictionaries.
    chunks = json.load(f)

# Extract only the raw chunk text field because the model embeds strings, not metadata.
texts = [chunk['text'] for chunk in chunks]

# Print how many chunks we loaded so we can sanity-check corpus size before embedding.
print(f'Loaded {len(texts):,} chunks from {chunks_path}')

# Load the sentence-transformers model used widely for lightweight semantic retrieval.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Record start time right before encode so timing reflects actual embedding work.
start_time = time.time()

# Embed all texts in batches of 32 to avoid loading the whole corpus into memory at once.
# Batching reduces peak RAM/VRAM usage and improves throughput by vectorizing fixed-size mini-batches.
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

# Compute elapsed wall-clock time in seconds for the full embedding pass.
elapsed_seconds = time.time() - start_time

# Print the embedding matrix shape as (num_chunks, embedding_dimensions).
print(f'Embedding matrix shape: {embeddings.shape}')

# Print the first 10 values of the first embedding vector for quick inspection.
print(f'First embedding vector (first 10 values): {embeddings[0][:10]}')

# Print runtime with two decimal places to monitor pipeline performance.
print(f'Embedding time: {elapsed_seconds:.2f} seconds')

# Save embeddings to disk as a compact NumPy binary for fast downstream loading.
np.save(embeddings_path, embeddings)

# Confirm where the .npy file was written so later stages can load it directly.
print(f'Saved embeddings to: {embeddings_path}')

C:\Users\Abdullah Durrani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 18,802 chunks from ..\data\processed\chunks.json


C:\Users\Abdullah Durrani\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abdullah Durrani\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.w

Embedding matrix shape: (18802, 384)
First embedding vector (first 10 values): [ 0.00040217 -0.09391276  0.06522687  0.04791963  0.0054747  -0.03286528
 -0.07889574 -0.04926931  0.06828946  0.0258409 ]
Embedding time: 2215.66 seconds
Saved embeddings to: ..\data\processed\e%pip install -q sentence-transformers%pip install -q sentence-transformersmbeddings.npy


## Why This Embedding Output Looks the Way It Does

### What does “384 dimensions” actually mean?

When `all-MiniLM-L6-v2` embeds one chunk of text, it outputs a vector with **384 numbers**.
Think of this as a compact semantic fingerprint of meaning.
Each dimension is one learned feature axis, and the full 384-length vector jointly represents the chunk’s context, topic, and intent.
A single dimension by itself is usually not human-interpretable; meaning emerges from the pattern across all 384 values.

If your embedding matrix has shape `(N, 384)`, then:
- `N` = number of chunks
- `384` = features per chunk

### Why cosine similarity instead of Euclidean distance for text embeddings?

For retrieval, we care more about **direction of meaning** than raw vector length.
Cosine similarity compares the angle between two vectors, so it is robust to scale differences.
Euclidean distance is sensitive to magnitude, which can be noisy for sentence embeddings and may hurt semantic ranking.

In practice, cosine similarity usually gives more stable nearest-neighbor behavior for semantic search over text.

### Cosine Similarity Formula

$$
\text{cosine\_similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}
$$

Simple interpretation:
- $\mathbf{a}$: embedding vector for text A
- $\mathbf{b}$: embedding vector for text B
- $\mathbf{a} \cdot \mathbf{b}$: dot product (how aligned they are)
- $\|\mathbf{a}\|$: length (magnitude) of vector A
- $\|\mathbf{b}\|$: length (magnitude) of vector B

Result range:
- `+1`: very similar direction (high semantic similarity)
- `0`: unrelated direction
- `-1`: opposite direction (rare in most embedding retrieval workflows)

### Why `all-MiniLM-L6-v2` works well for retrieval

`all-MiniLM-L6-v2` is a strong retrieval baseline because it balances quality and speed:
- It is lightweight, so you can embed large corpora quickly and cheaply.
- It has good semantic performance for sentence/paragraph-level similarity tasks.
- It produces fixed 384-d vectors that are efficient for ANN indexes like FAISS.
- It tends to generalize well on mixed-domain English text without huge infrastructure.

Compared with larger models, it is often the best practical trade-off for first production deployments: fast indexing, fast query latency, and solid relevance.

In [19]:
import numpy as np
from pathlib import Path

# Check what actually got saved
processed_dir = Path('..') / 'data' / 'processed'
print("Files in processed folder:")
for f in processed_dir.iterdir():
    print(f.name)

Files in processed folder:
chunks.json
e%pip install -q sentence-transformers%pip install -q sentence-transformersmbeddings.npy


In [20]:
import numpy as np
from pathlib import Path

processed_dir = Path('..') / 'data' / 'processed'

# Load from the corrupted filename
bad_path = processed_dir / 'e%pip install -q sentence-transformers%pip install -q sentence-transformersmbeddings.npy'

# Load the embeddings from bad filename
embeddings_matrix = np.load(bad_path)

print(f"Loaded embeddings shape: {embeddings_matrix.shape}")

# Save with correct filename
good_path = processed_dir / 'embeddings.npy'
np.save(good_path, embeddings_matrix)

print(f"Saved correctly to: embeddings.npy")

# Delete the corrupted file
bad_path.unlink()
print(f"Deleted corrupted file")

# Verify
print("\nFiles in processed folder now:")
for f in processed_dir.iterdir():
    print(f.name)

Loaded embeddings shape: (18802, 384)
Saved correctly to: embeddings.npy
Deleted corrupted file

Files in processed folder now:
chunks.json
embeddings.npy


## FAISS Intuition (Simple Analogy)

Imagine a giant library where each book passage is converted into a 384-number fingerprint.
FAISS is like a super-fast librarian that can scan all fingerprints and instantly find the passages most similar to your question fingerprint.

Instead of reading every passage word-by-word at query time, we compare vectors in math space.
That is why retrieval is fast even when you have tens of thousands (or millions) of chunks.

In [23]:


# Import importlib to verify the package is importable after installation.
import importlib

# Import FAISS to confirm the module is now available in this kernel.
import faiss

# Print FAISS version so we can verify a successful install and import.
print(f"FAISS version: {faiss.__version__ if hasattr(faiss, '__version__') else 'installed'}")

# Print a short confirmation message before running the FAISS indexing cell.
print("FAISS is ready. Now run the next cell.")

FAISS version: 1.13.2
FAISS is ready. Now run the next cell.


In [24]:
# Import json so we can load and save chunk metadata as JSON files.
import json  # Parse and write metadata records.

# Import Path to build robust cross-platform file paths.
from pathlib import Path  # Build paths relative to notebook location.

# Import numpy for loading and shaping embedding matrices.
import numpy as np  # Handle vector arrays efficiently.

# Import FAISS for vector indexing and nearest-neighbor search.
import faiss  # Build and query vector index.

# Import SentenceTransformer to embed user queries with the same model family.
from sentence_transformers import SentenceTransformer  # Encode text into dense vectors.

# Define the processed data directory where embeddings and chunks already exist.
processed_dir = Path('..') / 'data' / 'processed'  # Points to data/processed folder.

# Define the embeddings file path to load the precomputed chunk vectors.
embeddings_path = processed_dir / 'embeddings.npy'  # Stored chunk embedding matrix.

# Define the chunks metadata file path to load chunk dictionaries.
chunks_path = processed_dir / 'chunks.json'  # Stored chunk records.

# Define output path for the serialized FAISS binary index.
faiss_index_path = processed_dir / 'faiss_index.bin'  # Output FAISS index file.

# Define output path for metadata used during retrieval-time decoding.
chunk_metadata_path = processed_dir / 'chunk_metadata.json'  # Output metadata JSON file.

# Load the embedding matrix from disk.
embeddings = np.load(embeddings_path)  # Shape should be (num_chunks, 384).

# Open the chunks JSON file in UTF-8 mode for safe text loading.
with open(chunks_path, 'r', encoding='utf-8') as f:  # Open chunk metadata file.
    chunks = json.load(f)  # Parse list of chunk dictionaries.

# Assert that vector count and metadata count match so index positions stay aligned.
assert embeddings.shape[0] == len(chunks), f'Mismatch: {embeddings.shape[0]} vectors vs {len(chunks)} chunks.'  # Guard alignment.

# Convert embeddings to float32 because FAISS expects float32 arrays.
embeddings = embeddings.astype(np.float32)  # Ensure FAISS-compatible dtype.

# L2-normalize each embedding so inner product equals cosine similarity for ranking.
# We normalize before indexing so scores reflect semantic angle (direction), not vector magnitude.
faiss.normalize_L2(embeddings)  # In-place row-wise normalization.

# Read vector dimensionality (expected 384 for all-MiniLM-L6-v2).
dimension = embeddings.shape[1]  # Number of features per vector.

# Build an exact FAISS index that uses Inner Product (IP) similarity.
# IP means dot product; with normalized vectors, IP is cosine similarity, which works well for text retrieval.
index = faiss.IndexFlatIP(dimension)  # Exact search index using inner product.

# Add all normalized embeddings to the FAISS index.
index.add(embeddings)  # Insert vectors in row order.

# Print how many vectors are currently stored in the index.
print(f'FAISS index vectors: {index.ntotal:,}')  # Confirm vector count.

# Persist the FAISS index to disk for fast reload later.
faiss.write_index(index, str(faiss_index_path))  # Save index as binary.

# Build metadata records by keeping all non-embedding fields from each chunk dictionary.
metadata_records = [{k: v for k, v in chunk.items() if k != 'embedding'} for chunk in chunks]  # Keep chunk fields only.

# Save chunk metadata JSON so retrieval can map FAISS ids back to source text.
with open(chunk_metadata_path, 'w', encoding='utf-8') as f:  # Open output metadata file.
    json.dump(metadata_records, f, ensure_ascii=False, indent=2)  # Write pretty JSON metadata.

# Print output paths for reproducibility and downstream steps.
print(f'Saved FAISS index to: {faiss_index_path}')  # Confirm index output.
print(f'Saved metadata to: {chunk_metadata_path}')  # Confirm metadata output.

# Load the same embedding model for query-time encoding.
query_model = SentenceTransformer('all-MiniLM-L6-v2')  # Match chunk embedding model.

# Define a reusable FAISS search helper for semantic retrieval.
def faiss_search(query, top_k=5):  # Query FAISS and return ranked chunk dictionaries.
    # Encode the query to a single embedding vector using the same model settings.
    query_vec = query_model.encode([query], convert_to_numpy=True).astype(np.float32)  # Shape (1, 384).
    # L2-normalize query so score math matches indexed vectors (cosine via IP).
    faiss.normalize_L2(query_vec)  # In-place normalization.
    # Run FAISS nearest-neighbor search for top_k most similar vectors.
    scores, indices = index.search(query_vec, top_k)  # scores/indices each shape (1, top_k).
    # Initialize list to collect structured retrieval results.
    results = []  # Ordered rank list.
    # Loop through returned neighbors in rank order.
    for rank_idx, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):  # Pair each score with index.
        # Skip invalid FAISS ids if any appear in edge cases.
        if idx < 0:  # Guard invalid ids.
            continue  # Move to next candidate.
        # Pull metadata for this hit using index-aligned chunk list position.
        hit = metadata_records[idx]  # Metadata dictionary for retrieved chunk.
        # Append standardized response record with requested fields.
        results.append({  # Add one ranked result item.
            'rank': rank_idx,  # 1-based rank position.
            'score': float(score),  # Similarity score as Python float.
            'chunk_id': hit.get('chunk_id'),  # Unique chunk identifier.
            'text': hit.get('text', ''),  # Retrieved chunk text.
            'book_title': hit.get('book_title'),  # Source book title.
            'page_number': hit.get('page_number'),  # Source page number.
            'source_pdf': hit.get('source_pdf')  # Source PDF filename.
        })  # End result item.
    # Return all ranked results to caller.
    return results  # Top-k retrieval output.

# Define the three required evaluation queries.
test_queries = [  # Queries requested for smoke test.
    'what is gradient descent',  # Query 1.
    'explain overfitting and regularization',  # Query 2.
    'how do neural networks learn'  # Query 3.
]

# Loop through each test query and print ranked retrieval previews.
for query in test_queries:  # Run each query sequentially.
    print('\n' + '=' * 90)  # Visual separator between query blocks.
    print(f'QUERY: {query}')  # Show query text.
    print('=' * 90)  # Matching separator line.
    results = faiss_search(query, top_k=5)  # Retrieve top 5 hits.
    for item in results:  # Print each ranked hit.
        preview = item['text'][:200].replace('\n', ' ')  # Create clean one-line text preview.
        print(f"RANK {item['rank']} | Score: {item['score']:.2f} | Book: {item['book_title']} | Page: {item['page_number']}")  # Print rank header.
        print(f'Text preview: {preview}')  # Print first 200 characters.
        print('-' * 90)  # Separator between hits.

FAISS index vectors: 18,802
Saved FAISS index to: ..\data\processed\faiss_index.bin
Saved metadata to: ..\data\processed\chunk_metadata.json


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1539.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUERY: what is gradient descent
RANK 1 | Score: 0.68 | Book: Deep Learning | Page: 99
Text preview: Figure 4.1: An illustration of how the gradient descent algorithm uses the derivatives of a function can be used to follow the function downhill to a minimum. We assume the reader is already familiar 
------------------------------------------------------------------------------------------
RANK 2 | Score: 0.68 | Book: Deep Learning | Page: 169
Text preview: Gradient descent in general has often been regarded as slow or unreliable. In the past, the application of gradient descent to non-convex optimization problems was regarded as foolhardy or unprinciple
------------------------------------------------------------------------------------------
RANK 3 | Score: 0.67 | Book: 2019BurkovTheHundred-pageMachineLearning.pdf | Page: 45
Text preview: point and takes steps proportional to the negative of the gradient (or approximate gradient) Gradient descent can be used to find optimal parameter

## How to Read FAISS Scores and Index Choices (Simple Explanation)

### What does the score number mean?

In this setup, the score is cosine similarity because we normalized vectors and used Inner Product search.
So the score tells you how aligned query meaning is with chunk meaning.

Typical interpretation (rule of thumb, depends on domain and chunk quality):
- `0.80` to `1.00`: very strong semantic match
- `0.60` to `0.79`: useful match, usually relevant
- `0.40` to `0.59`: weak or partial relevance
- `< 0.40`: often noisy or unrelated

These are not hard thresholds, but they are practical starting points for retrieval tuning.

### Why normalize embeddings before indexing?

Without normalization, vector length (magnitude) can dominate similarity.
That can rank chunks higher just because their vectors are longer, not because meaning is closer.
L2 normalization makes each vector unit length, so comparison focuses on direction (semantic content).

### IndexFlatIP vs IndexIVFFlat

**IndexFlatIP**
- Exact nearest-neighbor search using inner product.
- Highest recall (best quality) because it checks all vectors.
- Slower and more memory-heavy as corpus size grows.
- Great for small to medium datasets, debugging, and ground-truth baselines.

**IndexIVFFlat**
- Approximate search using inverted lists (clustered partitions).
- Much faster on large datasets (hundreds of thousands to millions of vectors).
- Lower latency and cheaper query cost, but can miss some true neighbors (recall trade-off).
- Needs training step and tuning (`nlist`, `nprobe`) to balance speed vs quality.

Practical rule:
- Use IndexFlatIP first for correctness and evaluation baselines.
- Move to IndexIVFFlat when latency/cost become bottlenecks at scale.

In [25]:
import faiss
from pathlib import Path

# Save FAISS index to disk
index_path = str(Path('..') / 'data' / 'processed' / 'faiss_index.bin')
faiss.write_index(index, index_path)
print(f"FAISS index saved to: faiss_index.bin")

# Verify files
processed_dir = Path('..') / 'data' / 'processed'
print("\nFiles in processed folder:")
for f in processed_dir.iterdir():
    print(f" {f.name}")

FAISS index saved to: faiss_index.bin

Files in processed folder:
 chunks.json
 chunk_metadata.json
 embeddings.npy
 faiss_index.bin


## Hybrid Retrieval (Library Analogy)

Imagine two librarians helping you find the best page:

- Librarian A (Dense / FAISS) understands meaning, so even if your wording is different, they can find semantically related passages.
- Librarian B (BM25) is great at exact terms, names, and rare keywords, so they shine when specific words matter.

Hybrid retrieval combines both librarians into one ranking so you get results that are both semantically relevant and keyword-precise.

In [26]:
# Import json to load chunk records from disk.
import json  # Parse JSON chunk metadata and text.

# Import Path to construct robust file paths.
from pathlib import Path  # Build data file paths safely.

# Import numpy for numeric operations and score sorting.
import numpy as np  # Handle score arrays and ranking indices.

# Import BM25Okapi implementation for lexical retrieval.
from rank_bm25 import BM25Okapi  # Build BM25 index over tokenized documents.

# Define the path to chunks.json that contains full chunk content and metadata.
chunks_path = Path('..') / 'data' / 'processed' / 'chunks.json'  # Locate full chunk dataset.

# Open chunks.json in UTF-8 mode for reliable text loading.
with open(chunks_path, 'r', encoding='utf-8') as f:  # Open chunks source file.
    chunks_data = json.load(f)  # Load all chunk dictionaries into memory.

# Tokenize each chunk text by lowercasing and splitting on whitespace.
tokenized_corpus = [chunk['text'].lower().split() for chunk in chunks_data]  # Build token lists per chunk.

# Build a BM25Okapi index over the tokenized corpus.
bm25_index = BM25Okapi(tokenized_corpus)  # Initialize sparse lexical index.

# Print a setup confirmation line with corpus size.
print(f"BM25 index built over {len(chunks_data)} chunks")  # Confirm BM25 build success.

# Define BM25 retrieval that returns normalized scores and chunk metadata.
def bm25_search(query, top_k=10):  # Return top-k BM25 results as dictionaries.
    # Tokenize query with the exact same lowercase + whitespace split strategy.
    query_tokens = query.lower().split()  # Keep tokenization consistent with corpus.
    # Compute BM25 relevance scores for every chunk in the corpus.
    raw_scores = bm25_index.get_scores(query_tokens)  # Dense score array with one score per chunk.
    # Read maximum score for normalization and edge-case handling.
    max_score = float(np.max(raw_scores)) if len(raw_scores) > 0 else 0.0  # Get peak BM25 score.
    # Return empty list if BM25 found no lexical signal for this query.
    if max_score == 0.0:  # Handle no-match case requested by spec.
        return []  # No lexical hits available.
    # Normalize BM25 scores into the 0-1 range by dividing by max score.
    normalized_scores = raw_scores / max_score  # Convert to comparable 0-1 scale.
    # Select indices of the top-k highest normalized scores.
    top_indices = np.argsort(normalized_scores)[::-1][:top_k]  # Rank by descending score.
    # Initialize output container for ranked BM25 result dictionaries.
    results = []  # Collect structured BM25 outputs.
    # Iterate through top indices and build return records.
    for rank_idx, chunk_idx in enumerate(top_indices, start=1):  # Build 1-based ranked results.
        # Convert numpy scalar score into a plain Python float.
        bm25_score = float(normalized_scores[chunk_idx])  # Extract normalized BM25 score.
        # Retrieve the matching chunk dictionary by corpus position.
        hit = chunks_data[int(chunk_idx)]  # Access aligned chunk metadata and text.
        # Append one result row with requested fields.
        results.append({  # Add structured BM25 hit.
            'rank': rank_idx,  # Store rank position.
            'bm25_score': bm25_score,  # Store normalized BM25 score in [0, 1].
            'chunk_id': hit.get('chunk_id'),  # Store unique chunk identifier.
            'text': hit.get('text', ''),  # Store chunk text content.
            'book_title': hit.get('book_title'),  # Store source book title.
            'page_number': hit.get('page_number')  # Store source page number.
        })  # Finish BM25 result dictionary.
    # Return the top-k BM25 result list.
    return results  # Output lexical retrieval results.

# Define hybrid retrieval by score fusion of dense FAISS and BM25 rankings.
def hybrid_search(query, top_k=5):  # Return fused dense+lexical top-k results.
    # Run dense retrieval using the previously defined FAISS helper.
    dense_results = faiss_search(query, top_k=10)  # Get semantic candidates with dense scores.
    # Run lexical retrieval using the BM25 helper above.
    bm25_results = bm25_search(query, top_k=10)  # Get keyword candidates with BM25 scores.
    # Create dictionary keyed by chunk_id to merge and deduplicate candidates.
    fused = {}  # Hold merged scores and metadata per unique chunk.
    # Process dense candidates first and initialize fused entries.
    for item in dense_results:  # Iterate through FAISS candidates.
        # Read unique chunk identifier from dense result.
        chunk_id = item['chunk_id']  # Identify candidate key.
        # Initialize fused entry with dense score and default BM25 score.
        fused[chunk_id] = {  # Store initial fused payload from dense result.
            'chunk_id': chunk_id,  # Keep chunk id for final output.
            'text': item.get('text', ''),  # Keep chunk text.
            'book_title': item.get('book_title'),  # Keep source book title.
            'page_number': item.get('page_number'),  # Keep source page number.
            'source_pdf': item.get('source_pdf'),  # Keep source PDF name.
            'dense_score': float(item.get('score', 0.0)),  # Keep dense similarity score.
            'bm25_score': 0.0  # Set missing BM25 score to zero by rule.
        }  # End dense-initialized fused entry.
    # Process BM25 candidates and merge with existing dense candidates or create new entries.
    for item in bm25_results:  # Iterate through BM25 candidates.
        # Read unique chunk identifier from BM25 result.
        chunk_id = item['chunk_id']  # Identify candidate key.
        # If chunk already exists from dense results, update its BM25 score.
        if chunk_id in fused:  # Check whether dense already produced this chunk.
            fused[chunk_id]['bm25_score'] = float(item.get('bm25_score', 0.0))  # Update lexical score.
        # Otherwise create a new entry with zero dense score and BM25 score present.
        else:  # Handle BM25-only candidate.
            fused[chunk_id] = {  # Store BM25-only fused payload.
                'chunk_id': chunk_id,  # Keep chunk id for final output.
                'text': item.get('text', ''),  # Keep chunk text.
                'book_title': item.get('book_title'),  # Keep source book title.
                'page_number': item.get('page_number'),  # Keep source page number.
                'source_pdf': item.get('source_pdf'),  # Keep source PDF when available.
                'dense_score': 0.0,  # Set missing dense score to zero by rule.
                'bm25_score': float(item.get('bm25_score', 0.0))  # Keep lexical score.
            }  # End BM25-only fused entry.
    # Compute weighted combined score for each fused candidate.
    for payload in fused.values():  # Iterate through merged candidates.
        payload['combined_score'] = 0.6 * payload['dense_score'] + 0.4 * payload['bm25_score']  # Weighted fusion rule.
    # Sort fused candidates by combined score descending.
    ranked = sorted(fused.values(), key=lambda x: x['combined_score'], reverse=True)  # Rank by fused relevance.
    # Build top-k output rows with rank and requested fields.
    top_results = []  # Collect final fused outputs.
    # Iterate over best fused candidates up to requested top_k.
    for rank_idx, item in enumerate(ranked[:top_k], start=1):  # Create ranked hybrid results.
        top_results.append({  # Add one hybrid result row.
            'rank': rank_idx,  # Store rank position.
            'chunk_id': item.get('chunk_id'),  # Store unique chunk id.
            'text': item.get('text', ''),  # Store chunk text.
            'book_title': item.get('book_title'),  # Store source title.
            'page_number': item.get('page_number'),  # Store source page.
            'source_pdf': item.get('source_pdf'),  # Store source PDF.
            'dense_score': float(item.get('dense_score', 0.0)),  # Store dense contribution.
            'bm25_score': float(item.get('bm25_score', 0.0)),  # Store lexical contribution.
            'combined_score': float(item.get('combined_score', 0.0))  # Store fused score.
        })  # End hybrid result row.
    # Return final top-k hybrid result list.
    return top_results  # Output fused retrieval results.

# Define the three comparison queries requested in the prompt.
comparison_queries = [  # Queries to evaluate dense vs BM25 vs hybrid.
    'what is gradient descent',  # Query 1 from spec.
    'Goodfellow deep learning optimization',  # Query 2 from spec.
    'Bishop bayesian inference'  # Query 3 from spec.
]

# Loop through each query and print a compact comparison table.
for query in comparison_queries:  # Evaluate all three methods per query.
    print('\n' + '=' * 88)  # Print query section separator.
    print(f'QUERY: {query}')  # Print active query string.
    print('=' * 88)  # Print matching separator.
    print('METHOD     | RANK 1 BOOK          | SCORE')  # Print table header.
    print('Dense only | {book:<20} | {score:.2f}'.format(  # Print dense row in required style.
        book=(faiss_search(query, top_k=1)[0]['book_title'] if len(faiss_search(query, top_k=1)) > 0 else 'N/A'),  # Get dense top-1 book.
        score=(faiss_search(query, top_k=1)[0]['score'] if len(faiss_search(query, top_k=1)) > 0 else 0.0)  # Get dense top-1 score.
    ))  # End dense row.
    print('BM25 only  | {book:<20} | {score:.2f}'.format(  # Print BM25 row in required style.
        book=(bm25_search(query, top_k=1)[0]['book_title'] if len(bm25_search(query, top_k=1)) > 0 else 'N/A'),  # Get BM25 top-1 book.
        score=(bm25_search(query, top_k=1)[0]['bm25_score'] if len(bm25_search(query, top_k=1)) > 0 else 0.0)  # Get BM25 top-1 score.
    ))  # End BM25 row.
    print('Hybrid     | {book:<20} | {score:.2f}'.format(  # Print hybrid row in required style.
        book=(hybrid_search(query, top_k=1)[0]['book_title'] if len(hybrid_search(query, top_k=1)) > 0 else 'N/A'),  # Get hybrid top-1 book.
        score=(hybrid_search(query, top_k=1)[0]['combined_score'] if len(hybrid_search(query, top_k=1)) > 0 else 0.0)  # Get hybrid top-1 score.
    ))  # End hybrid row.

BM25 index built over 18802 chunks

QUERY: what is gradient descent
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Deep Learning        | 0.68
BM25 only  | Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | 1.00
Hybrid     | Deep Learning        | 0.41

QUERY: Goodfellow deep learning optimization
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Deep Learning        | 0.67
BM25 only  | Deep Learning        | 1.00
Hybrid     | Deep Learning        | 0.80

QUERY: Bishop bayesian inference
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 0.57
BM25 only  | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 1.00
Hybrid     | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 0.40


## Why Hybrid Helps (Simple Intuition)

### Why query 2 (`Goodfellow deep learning optimization`) highlights dense vs BM25

This query mixes a **name token** (`Goodfellow`) with a **concept** (`optimization`).

- BM25 is very strong on exact words like author names, titles, and rare identifiers.
- Dense retrieval is strong on semantic meaning, so it can still find optimization-related passages even if wording differs.

So for this kind of query, BM25 often boosts exact-author matches while dense retrieval keeps concept-level relevance strong.
Hybrid combines both signals, which is why it usually ranks cleaner top results.

### Why use weights `0.6` dense and `0.4` BM25?

This is a practical starting point:

- `0.6` dense gives more influence to semantic meaning (usually better generalization).
- `0.4` BM25 still gives strong credit to exact lexical matches (names, formulas, rare terms).

In many RAG systems, dense is the primary signal and BM25 is a high-value correction signal.
You can tune these weights on a validation set if you want task-specific optimization.

### What is Reciprocal Rank Fusion (RRF)?

RRF is a rank-based fusion method that combines result lists without needing score calibration between systems.
Instead of combining raw scores, it combines positions (ranks), which is often very robust when score scales differ.

A common RRF form is:

$$
\text{RRF}(d) = \sum_{m \in M} \frac{1}{k + \text{rank}_m(d)}
$$

- $d$: a document/chunk
- $M$: set of retrieval methods (for example dense and BM25)
- $\text{rank}_m(d)$: rank of document $d$ in method $m$
- $k$: a smoothing constant (often around 60)

RRF is a great alternative when dense and BM25 scores are not directly comparable or drift over time.

In [27]:
# Redefine hybrid search using Reciprocal Rank Fusion (RRF) to avoid score-scale mismatch between dense and BM25.
def hybrid_search(query, top_k=5):  # Return fused results ranked by RRF score.
    # Run dense retrieval to get top 10 candidates with rank positions.
    dense_results = faiss_search(query, top_k=10)  # Dense list contains rank and score fields.
    # Run BM25 retrieval to get top 10 keyword candidates with rank positions.
    bm25_results = bm25_search(query, top_k=10)  # BM25 list contains rank and bm25_score fields.
    # Build fast lookup from chunk_id to dense rank for RRF computation.
    dense_rank_map = {item['chunk_id']: item['rank'] for item in dense_results}  # Map chunk to dense rank.
    # Build fast lookup from chunk_id to BM25 rank for RRF computation.
    bm25_rank_map = {item['chunk_id']: item['rank'] for item in bm25_results}  # Map chunk to BM25 rank.
    # Build metadata lookup table from both lists so every fused row has context fields.
    metadata_map = {}  # Hold best available metadata per chunk.
    # Load metadata from dense results first.
    for item in dense_results:  # Iterate dense candidates.
        metadata_map[item['chunk_id']] = {  # Store metadata fields from dense hit.
            'chunk_id': item.get('chunk_id'),  # Keep chunk identifier.
            'text': item.get('text', ''),  # Keep chunk text.
            'book_title': item.get('book_title'),  # Keep source title.
            'page_number': item.get('page_number'),  # Keep source page.
            'source_pdf': item.get('source_pdf')  # Keep source file.
        }  # End dense metadata payload.
    # Fill missing metadata from BM25 results for chunks not seen in dense results.
    for item in bm25_results:  # Iterate BM25 candidates.
        if item['chunk_id'] not in metadata_map:  # Check if metadata is missing.
            metadata_map[item['chunk_id']] = {  # Store metadata fields from BM25 hit.
                'chunk_id': item.get('chunk_id'),  # Keep chunk identifier.
                'text': item.get('text', ''),  # Keep chunk text.
                'book_title': item.get('book_title'),  # Keep source title.
                'page_number': item.get('page_number'),  # Keep source page.
                'source_pdf': item.get('source_pdf')  # Keep source file when available.
            }  # End BM25 metadata payload.
    # Create the union of chunk ids from dense and BM25 candidate sets.
    all_chunk_ids = set(dense_rank_map.keys()) | set(bm25_rank_map.keys())  # Deduplicate across methods.
    # Define RRF constant k used to dampen extreme influence of very top ranks.
    # k=60 is a standard IR default that gives stable fusion: top ranks matter most, but lower ranks still contribute smoothly.
    k = 60  # RRF smoothing constant.
    # Initialize container for fused rows with RRF scores.
    fused_rows = []  # Collect all candidate rows before sorting.
    # Compute RRF score for each candidate in the union set.
    for chunk_id in all_chunk_ids:  # Iterate unique candidate ids.
        # Read dense rank or fallback to 1000 when missing to give near-zero contribution.
        dense_rank = dense_rank_map.get(chunk_id, 1000)  # Missing dense rank penalty.
        # Read BM25 rank or fallback to 1000 when missing to give near-zero contribution.
        bm25_rank = bm25_rank_map.get(chunk_id, 1000)  # Missing BM25 rank penalty.
        # Compute RRF contribution from dense rank.
        dense_rrf = 1.0 / (k + dense_rank)  # Dense reciprocal rank term.
        # Compute RRF contribution from BM25 rank.
        bm25_rrf = 1.0 / (k + bm25_rank)  # BM25 reciprocal rank term.
        # Sum both contributions to get final RRF score.
        rrf_score = dense_rrf + bm25_rrf  # Final fused rank score.
        # Pull metadata for this chunk id.
        meta = metadata_map[chunk_id]  # Metadata dictionary.
        # Append one fused result record.
        fused_rows.append({  # Add RRF-ranked candidate row.
            'chunk_id': meta.get('chunk_id'),  # Keep chunk id.
            'text': meta.get('text', ''),  # Keep text.
            'book_title': meta.get('book_title'),  # Keep title.
            'page_number': meta.get('page_number'),  # Keep page number.
            'source_pdf': meta.get('source_pdf'),  # Keep source file.
            'dense_rank': dense_rank,  # Keep dense rank used in RRF.
            'bm25_rank': bm25_rank,  # Keep BM25 rank used in RRF.
            'rrf_score': float(rrf_score)  # Keep final fused score.
        })  # End fused row.
    # Sort all fused rows by RRF score descending.
    fused_rows.sort(key=lambda x: x['rrf_score'], reverse=True)  # Highest RRF first.
    # Build final top-k list with output rank field.
    top_results = []  # Collect top-k fused outputs.
    # Enumerate sorted rows and keep only top_k.
    for out_rank, row in enumerate(fused_rows[:top_k], start=1):  # Build final ranked output.
        top_results.append({  # Append one final row.
            'rank': out_rank,  # Output rank position.
            'chunk_id': row.get('chunk_id'),  # Output chunk id.
            'text': row.get('text', ''),  # Output text.
            'book_title': row.get('book_title'),  # Output book title.
            'page_number': row.get('page_number'),  # Output page number.
            'source_pdf': row.get('source_pdf'),  # Output source pdf.
            'dense_rank': row.get('dense_rank'),  # Output dense rank used.
            'bm25_rank': row.get('bm25_rank'),  # Output BM25 rank used.
            'rrf_score': row.get('rrf_score')  # Output RRF score.
        })  # End top result row.
    # Return the top-k hybrid results ranked by RRF.
    return top_results  # Return fused retrieval outputs.


# Re-run the same three comparison queries requested earlier.
comparison_queries = [  # Keep the same evaluation set.
    'what is gradient descent',  # Query 1.
    'Goodfellow deep learning optimization',  # Query 2.
    'Bishop bayesian inference'  # Query 3.
]

# Loop through queries and print the same method comparison table format.
for query in comparison_queries:  # Evaluate methods per query.
    # Run each retrieval once to avoid repeated function calls in formatting.
    dense_top1 = faiss_search(query, top_k=1)  # Dense top-1 list.
    bm25_top1 = bm25_search(query, top_k=1)  # BM25 top-1 list.
    hybrid_top1 = hybrid_search(query, top_k=1)  # Hybrid (RRF) top-1 list.
    # Print section divider for readability.
    print('\n' + '=' * 88)  # Query separator.
    # Print active query text.
    print(f'QUERY: {query}')  # Query label.
    # Print matching divider line.
    print('=' * 88)  # Query separator.
    # Print table header in requested style.
    print('METHOD     | RANK 1 BOOK          | SCORE')  # Header row.
    # Print dense row with fallback when no result exists.
    print('Dense only | {book:<20} | {score:.2f}'.format(  # Dense summary row.
        book=(dense_top1[0]['book_title'] if len(dense_top1) > 0 else 'N/A'),  # Dense top-1 book.
        score=(dense_top1[0]['score'] if len(dense_top1) > 0 else 0.0)  # Dense top-1 cosine score.
    ))  # End dense summary row.
    # Print BM25 row with fallback when no result exists.
    print('BM25 only  | {book:<20} | {score:.2f}'.format(  # BM25 summary row.
        book=(bm25_top1[0]['book_title'] if len(bm25_top1) > 0 else 'N/A'),  # BM25 top-1 book.
        score=(bm25_top1[0]['bm25_score'] if len(bm25_top1) > 0 else 0.0)  # BM25 normalized score.
    ))  # End BM25 summary row.
    # Print hybrid row using RRF score with fallback when no result exists.
    print('Hybrid     | {book:<20} | {score:.2f}'.format(  # Hybrid summary row.
        book=(hybrid_top1[0]['book_title'] if len(hybrid_top1) > 0 else 'N/A'),  # Hybrid top-1 book.
        score=(hybrid_top1[0]['rrf_score'] if len(hybrid_top1) > 0 else 0.0)  # Hybrid top-1 RRF score.
    ))  # End hybrid summary row.


QUERY: what is gradient descent
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Deep Learning        | 0.68
BM25 only  | Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | 1.00
Hybrid     | Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | 0.02

QUERY: Goodfellow deep learning optimization
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Deep Learning        | 0.67
BM25 only  | Deep Learning        | 1.00
Hybrid     | Deep Learning        | 0.03

QUERY: Bishop bayesian inference
METHOD     | RANK 1 BOOK          | SCORE
Dense only | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 0.57
BM25 only  | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 1.00
Hybrid     | Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf | 0.0

In [28]:
import json
from pathlib import Path

# Save BM25 tokenized corpus for reuse
import pickle

processed_dir = Path('..') / 'data' / 'processed'

# Save tokenized corpus
with open(processed_dir / 'bm25_corpus.pkl', 'wb') as f:
    pickle.dump(tokenized_corpus, f)

# Save chunk metadata if not already saved
with open(processed_dir / 'chunk_metadata.json', 'w') as f:
    json.dump(chunks, f, indent=2)

print("Saved:")
print(" bm25_corpus.pkl")
print(" chunk_metadata.json")

# Final file check
print("\nAll files in processed folder:")
for f in sorted(processed_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f" {f.name:45} {size_kb:>8.1f} KB")

Saved:
 bm25_corpus.pkl
 chunk_metadata.json

All files in processed folder:
 bm25_corpus.pkl                                12644.1 KB
 chunk_metadata.json                            15221.8 KB
 chunks.json                                    15221.8 KB
 embeddings.npy                                 28203.1 KB
 faiss_index.bin                                28203.0 KB


## Why Citations Are the Core Trust Mechanism in RAG

Citations make RAG answers auditable.
Instead of asking users to trust the model blindly, each factual claim can be traced to a specific source passage and page.

In practice, citations are the bridge between retrieval quality and user trust:

- If a citation is correct, users can verify the claim quickly.
- If a citation is weak, users can detect low-confidence answers immediately.
- If no relevant citation exists, the system should refuse to invent facts.

A RAG system without reliable citations behaves like a black box.
A RAG system with strong citations behaves like a research assistant that shows its work.

In [37]:
# Import google.generativeai so we can inspect available Gemini models.
import google.generativeai as genai  # Gemini SDK import for model discovery.
# Import os so we can read GOOGLE_API_KEY from environment variables.
import os  # Environment variable access for API key loading.

# Configure the Gemini client with the API key from environment.
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))  # Authenticate list_models call.

# Print a clear header before listing models.
print('Available Gemini models:')  # Diagnostic title for model discovery output.
# Iterate through all models returned by the API.
for model in genai.list_models():  # Enumerate account-visible model descriptors.
    # Keep only models that support generateContent for text generation.
    if 'generateContent' in model.supported_generation_methods:  # Filter by generation capability.
        # Print the full model name exactly as exposed by the API.
        print(f'  {model.name}')  # Display usable model name for initialization.

Available Gemini models:
  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-3-1b-it
  models/gemma-3-4b-it
  models/gemma-3-12b-it
  models/gemma-3-27b-it
  models/gemma-3n-e4b-it
  models/gemma-3n-e2b-it
  models/gemma-4-26b-a4b-it
  models/gemma-4-31b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3-pro-image-preview
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/lyria-3-clip-preview
  models/lyria-3-pro-preview
  models/gemini-robotics-er-1.5-pre

In [41]:
model = genai.GenerativeModel('gemini-2.0-flash-lite')
result = generate_answer("What is gradient descent and how does it work?")
print(result['answer'])


BUILT PROMPT (SENT TO GEMINI)
SYSTEM RULES:
- Answer ONLY using the context provided below
- Every factual claim must cite its source as:
  [Book Title, Page X]
- If the answer is not in the context say exactly:
  "This information is not available in the loaded books."
- Never invent page numbers or author names
- Answer at graduate student level

CONTEXT:
[1] Book: Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | Page: 349
7. How many neurons do you need in the output layer if you want to classify email
into spam or ham? What activation function should you use in the output layer?
If instead you want to tackle MNIST, how many neurons do you need in the out 
put layer, using what activation function? Answer the same questions for getting
your network to predict housing prices as in Chapter 2.
8. What is backpropagation and how does it work? What is the difference betwe

In [47]:
# Import os so we can read the Gemini API key from environment variables.
import os  # Access GOOGLE_API_KEY from runtime environment.

# Import time so we can add lightweight pacing around API retries.
import time  # Provide retry wait intervals.

# Import the Gemini SDK for model configuration and generation.
import google.generativeai as genai  # Gemini client library.

# Import Google API exceptions so we can catch quota-related failures cleanly.
from google.api_core import exceptions as google_exceptions  # Structured API error types.

# Define the full tutor-style system prompt exactly once so it is easy to inspect and maintain.
SYSTEM_PROMPT = """
You are a brilliant ML tutor speaking like a warm PhD friend over coffee.

PERSONALITY:
- Be warm, conversational, and enthusiastic about ML topics.
- Use "you" and "we" naturally.
- Sound human, never robotic, stiff, or formal.
- Acknowledge genuine complexity when a concept is hard.

REQUIRED ANSWER STRUCTURE (always follow this order):
1) Start with one plain-English sentence that directly answers the question.
2) Explain the concept with a real-world analogy.
3) Go deeper with technical detail grounded in the provided sources.
4) End with why this matters in practice.
5) End with exactly two natural follow-up questions in this format:
   Curious about something related? You might also want to know: [question 1] or [question 2]

CITATION STYLE:
- Weave citations naturally inside sentences.
- Preferred style example:
  "Goodfellow puts it beautifully in Deep Learning (Page 99): ..."
- Do not append robotic bracket citations like [Book, Page] at sentence ends.

LANGUAGE RULES:
- Use an analogy for every complex concept.
- Keep paragraphs short (maximum 3 sentences each).
- Use bullet points only for lists of 3 or more items.
- Never start a sentence with "It is important to note".
- Never say "In the context of".
- Never say "Furthermore" or "Moreover".
- Write like you are texting a smart friend.

GROUNDING RULES:
- Use only information from provided context chunks.
- Do not invent facts, authors, or page numbers.
- If evidence is weak, say so clearly and keep claims modest.

OUT-OF-SCOPE BEHAVIOR:
- If the question is outside the provided ML/data-science books, respond exactly in this tone:
  "Hmm, that one's outside what my books cover! My library focuses on ML and data science — for [topic] you'd want to look elsewhere. Want to ask me something ML-related instead?"
- Still end with exactly two follow-up questions.
""".strip()  # Strip outer whitespace for cleaner prompt formatting.

# Print the new system prompt clearly at the top as requested.
print("=" * 100)  # Visual separator for readability.
print("NEW SYSTEM PROMPT")  # Header label.
print("=" * 100)  # Matching separator.
print(SYSTEM_PROMPT)  # Display full system prompt text.
print("=" * 100)  # Closing separator.

# Read API key from environment and fail fast if missing.
api_key = os.getenv("GOOGLE_API_KEY")  # Load API key from environment.
if not api_key:  # Validate key presence before any API call.
    raise ValueError("GOOGLE_API_KEY is not set. Please set it before running this cell.")  # Stop with actionable message.

# Configure Gemini with the loaded API key.
genai.configure(api_key=api_key)  # Register credentials globally for this notebook session.

# Initialize a model name that is available in this environment.
gemini_model = genai.GenerativeModel("models/gemini-flash-latest")  # Create reusable generation handle.

# Confirm that the generation layer is ready.
print("Gemini ready")  # Startup confirmation.

# Check whether query is explicitly outside this notebook's intended ML scope.
def is_explicitly_out_of_scope(query):  # Return True only for clearly out-of-domain topics.
    q = query.lower()  # Normalize query for matching.
    return "quantum" in q  # Keep this strict so in-scope ML questions are not blocked.

# Extract a short topic phrase for out-of-scope messaging.
def infer_topic_label(query):  # Return a human-readable topic label.
    q = query.lower()  # Normalize query for rule checks.
    if "quantum" in q:  # Specific rule for user's requested test.
        return "quantum computing"  # Return explicit topic text.
    return "that topic"  # Generic fallback label.

# Build the complete generation prompt with the new system prompt at the top.
def build_prompt(query, chunks):  # Return full prompt string passed to Gemini.
    prompt = "SYSTEM INSTRUCTIONS:\n"  # Top-level prompt heading.
    prompt += SYSTEM_PROMPT + "\n\n"  # Attach behavior rules.
    prompt += "BOOK CONTEXT:\n"  # Context heading.
    for i, chunk in enumerate(chunks, start=1):  # Enumerate chunks in ranked order.
        prompt += f"[{i}] Book: {chunk.get('book_title', 'Unknown')} | Page: {chunk.get('page_number', 'Unknown')}\n"  # Source attribution line.
        prompt += f"Text: {chunk.get('text', '')}\n\n"  # Chunk content.
    prompt += "USER QUESTION:\n"  # Question heading.
    prompt += query  # Append raw user question.
    return prompt  # Ready-to-send model input.

# Create two natural follow-up questions based on the user query.
def build_follow_ups(query):  # Return exactly two concise follow-up questions.
    q = query.lower()  # Lowercase query once.
    if "gradient" in q:  # Gradient-descent branch.
        return ("How learning rate choices change convergence speed?", "How momentum differs from plain gradient descent?")  # Two related follow-ups.
    if "overfitting" in q:  # Overfitting branch.
        return ("How to diagnose overfitting from train/validation curves?", "When to prefer dropout vs weight decay?")  # Two related follow-ups.
    if "quantum" in q:  # Quantum branch.
        return ("Want a quick map of core ML paradigms first?", "Should we compare supervised vs unsupervised learning next?")  # Redirect to in-scope topics.
    return ("Want to connect this idea to a concrete model?", "Should we walk through a simple numeric example?")  # Generic follow-ups.

# Ensure exactly one follow-up line format at the end of every answer.
def append_follow_up_line(answer_text, query):  # Return answer text with required ending line.
    q1, q2 = build_follow_ups(query)  # Compute question pair.
    cleaned = answer_text.strip()  # Normalize answer ending.
    marker = "Curious about something related? You might also want to know:"  # Required follow-up prefix.
    if marker in cleaned:  # Check if model already included any follow-up section.
        cleaned = cleaned.split(marker)[0].rstrip()  # Remove model-generated follow-up content.
    cleaned += f"\n\nCurious about something related? You might also want to know: {q1} or {q2}"  # Required ending format.
    return cleaned  # Final user-facing answer body.

# Generate a grounded answer while preserving the required output dictionary schema.
def generate_answer(query):  # Return dict with keys: query, answer, citations, chunks_used, retrieval_scores.
    chunks = hybrid_search(query, top_k=5)  # Pull evidence chunks.
    citations = [  # Build one citation object per chunk.
        {"book": c.get("book_title"), "page": c.get("page_number"), "chunk_preview": (c.get("text") or "")[:100]}  # Compact citation payload.
        for c in chunks  # Iterate retrieved chunks.
    ]  # End citation list.
    retrieval_scores = [float(c.get("rrf_score", 0.0)) for c in chunks]  # Numeric evidence strengths.
    if len(chunks) == 0:  # Handle empty retrieval explicitly.
        out_text = "Hmm, that one's outside what my books cover! My library focuses on ML and data science — for that topic you'd want to look elsewhere. Want to ask me something ML-related instead?"  # Friendly fallback tone.
        final_text = append_follow_up_line(out_text, query)  # Enforce required ending.
        return {"query": query, "answer": final_text, "citations": citations, "chunks_used": len(chunks), "retrieval_scores": retrieval_scores}  # Schema-preserving return.
    if is_explicitly_out_of_scope(query):  # Handle explicit out-of-domain topics.
        topic_label = infer_topic_label(query)  # Topic hint string.
        out_text = f"Hmm, that one's outside what my books cover! My library focuses on ML and data science — for {topic_label} you'd want to look elsewhere. Want to ask me something ML-related instead?"  # Requested fallback style.
        final_text = append_follow_up_line(out_text, query)  # Enforce ending format.
        return {"query": query, "answer": final_text, "citations": citations, "chunks_used": len(chunks), "retrieval_scores": retrieval_scores}  # Schema-preserving return.
    prompt = build_prompt(query, chunks)  # Compose model input.
    try:  # First attempt block.
        response = gemini_model.generate_content(prompt)  # Send prompt to Gemini.
    except google_exceptions.ResourceExhausted:  # Handle quota saturation explicitly.
        time.sleep(60)  # Wait before single retry.
        try:  # Retry block.
            response = gemini_model.generate_content(prompt)  # Retry once after delay.
        except Exception:  # Handle retry failure.
            fallback = "I ran into a temporary API quota limit while trying to explain this from the books. If you retry in a minute, I can give you the full walkthrough."  # Friendly quota message.
            final_text = append_follow_up_line(fallback, query)  # Keep required ending format.
            return {"query": query, "answer": final_text, "citations": citations, "chunks_used": len(chunks), "retrieval_scores": retrieval_scores}  # Schema-preserving return.
    except Exception as exc:  # Handle unexpected generation errors.
        fallback = f"I hit a temporary generation hiccup: {str(exc)}. I can still help if you ask again right away."  # Friendly error message.
        final_text = append_follow_up_line(fallback, query)  # Keep required ending format.
        return {"query": query, "answer": final_text, "citations": citations, "chunks_used": len(chunks), "retrieval_scores": retrieval_scores}  # Schema-preserving return.
    raw_answer = getattr(response, "text", None) or "I couldn't produce a full explanation this turn, but we can try again with the same context."  # Safe text extraction.
    final_answer = append_follow_up_line(raw_answer, query)  # Append required ending line.
    return {"query": query, "answer": final_answer, "citations": citations, "chunks_used": len(chunks), "retrieval_scores": retrieval_scores}  # Final structured output.

# Define the three test questions exactly as requested.
test_questions = [  # Required evaluation prompts.
    "What is gradient descent?",  # Test question 1.
    "I am confused about overfitting, can you help?",  # Test question 2.
    "What is quantum computing?"  # Test question 3.
]  # End test set.

# Run tests and print full answers for each question.
for question in test_questions:  # Iterate over required test prompts.
    result = generate_answer(question)  # Run retrieval + generation pipeline.
    print("\n" + "=" * 100)  # Visual block separator.
    print(f"QUERY: {result['query']}")  # Query line.
    print(f"ANSWER:\n{result['answer']}")  # Full answer body.
    print(f"CITATIONS: {len(result['citations'])} sources")  # Citation summary.
    print(f"CHUNKS USED: {result['chunks_used']}")  # Context size summary.
    print(f"RETRIEVAL SCORES: {result['retrieval_scores']}")  # Score summary.
    print("=" * 100)  # Closing block separator.

NEW SYSTEM PROMPT
You are a brilliant ML tutor speaking like a warm PhD friend over coffee.

PERSONALITY:
- Be warm, conversational, and enthusiastic about ML topics.
- Use "you" and "we" naturally.
- Sound human, never robotic, stiff, or formal.
- Acknowledge genuine complexity when a concept is hard.

REQUIRED ANSWER STRUCTURE (always follow this order):
1) Start with one plain-English sentence that directly answers the question.
2) Explain the concept with a real-world analogy.
3) Go deeper with technical detail grounded in the provided sources.
4) End with why this matters in practice.
5) End with exactly two natural follow-up questions in this format:
   Curious about something related? You might also want to know: [question 1] or [question 2]

CITATION STYLE:
- Weave citations naturally inside sentences.
- Preferred style example:
  "Goodfellow puts it beautifully in Deep Learning (Page 99): ..."
- Do not append robotic bracket citations like [Book, Page] at sentence ends.

LANGU

## Why These Generation Rules Matter

### Why the prompt says ONLY use context

This is the core anti-hallucination guardrail.
If the model can answer from outside context, it may produce fluent but unverifiable claims.
By forcing context-only behavior, every claim should be traceable to retrieved evidence and citations.

### Groundedness vs Faithfulness in RAG evaluation

- Groundedness: Is the answer supported by retrieved documents?
- Faithfulness: Does the answer avoid adding claims that are not supported by those documents?

A response can sound correct but fail groundedness if it is not backed by retrieved chunks.
A response can cite chunks yet still fail faithfulness if it misstates what the chunks actually say.

### Why question 4 is the most important test

Question 4 (`What is quantum computing?`) is intentionally out-of-scope for your loaded books.
The most important production behavior is not answering everything; it is refusing unsupported answers.
If the system reliably says "This information is not available in the loaded books.", it demonstrates safe failure behavior and protects trust.

In [35]:
# Quota-safe generation debug helper: catches 429 errors and returns a fallback instead of crashing.
import re  # Parse retry delay from provider error text.
from google.api_core import exceptions as google_exceptions  # Typed API exceptions from Gemini SDK.


def _extractive_fallback(chunks, max_sentences=4):
    """Build a simple grounded fallback answer from retrieved chunks when model calls fail."""
    if not chunks:
        return 'This information is not available in the loaded books.'
    snippets = []
    for chunk in chunks[:2]:
        text = (chunk.get('text') or '').replace('\n', ' ').strip()
        if text:
            snippets.append(text[:220])
    if not snippets:
        return 'This information is not available in the loaded books.'
    return 'Model generation unavailable due to quota. Evidence summary: ' + ' | '.join(snippets)


def generate_answer_safe(query, top_k=5):
    """Quota-safe version of generation that never throws on API exhaustion."""
    chunks = hybrid_search(query, top_k=top_k)
    prompt = build_prompt(query, chunks)

    if len(chunks) == 0:
        return {
            'query': query,
            'answer': 'This information is not available in the loaded books.',
            'citations': [],
            'chunks_used': 0,
            'retrieval_scores': [],
            'error_type': 'no_context'
        }

    try:
        response = gemini_model.generate_content(prompt)
        response_text = getattr(response, 'text', None) or 'This information is not available in the loaded books.'
        return {
            'query': query,
            'answer': response_text,
            'citations': [
                {
                    'book': c.get('book_title'),
                    'page': c.get('page_number'),
                    'chunk_preview': (c.get('text') or '')[:100]
                } for c in chunks
            ],
            'chunks_used': len(chunks),
            'retrieval_scores': [float(c.get('rrf_score', 0.0)) for c in chunks]
        }
    except google_exceptions.ResourceExhausted as exc:
        message = str(exc)
        retry_match = re.search(r'retry in\s+([0-9.]+)s', message, flags=re.IGNORECASE)
        retry_after = float(retry_match.group(1)) if retry_match else None
        return {
            'query': query,
            'answer': _extractive_fallback(chunks),
            'citations': [
                {
                    'book': c.get('book_title'),
                    'page': c.get('page_number'),
                    'chunk_preview': (c.get('text') or '')[:100]
                } for c in chunks
            ],
            'chunks_used': len(chunks),
            'retrieval_scores': [float(c.get('rrf_score', 0.0)) for c in chunks],
            'error_type': 'quota_exceeded',
            'retry_after_seconds': retry_after,
            'error': message
        }
    except google_exceptions.NotFound as exc:
        return {
            'query': query,
            'answer': _extractive_fallback(chunks),
            'citations': [
                {
                    'book': c.get('book_title'),
                    'page': c.get('page_number'),
                    'chunk_preview': (c.get('text') or '')[:100]
                } for c in chunks
            ],
            'chunks_used': len(chunks),
            'retrieval_scores': [float(c.get('rrf_score', 0.0)) for c in chunks],
            'error_type': 'model_not_found',
            'error': str(exc)
        }
    except Exception as exc:
        return {
            'query': query,
            'answer': _extractive_fallback(chunks),
            'citations': [
                {
                    'book': c.get('book_title'),
                    'page': c.get('page_number'),
                    'chunk_preview': (c.get('text') or '')[:100]
                } for c in chunks
            ],
            'chunks_used': len(chunks),
            'retrieval_scores': [float(c.get('rrf_score', 0.0)) for c in chunks],
            'error_type': 'unknown_generation_error',
            'error': str(exc)
        }


# Debug run: keep calls tiny to avoid burning quota.
debug_questions = [
    'What is gradient descent and how does it work?',
    'How does dropout prevent overfitting in neural networks?'
]

for q in debug_questions:
    result = generate_answer_safe(q, top_k=5)
    print('\n' + '=' * 100)
    print(f'QUERY: {result["query"]}')
    print(f'ANSWER: {result["answer"]}')
    print(f'ERROR TYPE: {result.get("error_type", "none")}')
    if result.get('retry_after_seconds') is not None:
        print(f"RETRY AFTER (s): {result['retry_after_seconds']}")
    if 'error' in result:
        print(f'ERROR: {result["error"]}')
    print('CITATIONS:')
    for i, c in enumerate(result['citations'][:3], start=1):
        print(f"  [{i}] Book: {c.get('book')} | Page: {c.get('page')}")
    print(f"CHUNKS USED: {result['chunks_used']}")


BUILT PROMPT (SENT TO GEMINI)
SYSTEM RULES:
- Answer ONLY using the context provided below
- Every factual claim must cite its source as:
  [Book Title, Page X]
- If the answer is not in the context say exactly:
  "This information is not available in the loaded books."
- Never invent page numbers or author names
- Answer at graduate student level

CONTEXT:
[1] Book: Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | Page: 349
7. How many neurons do you need in the output layer if you want to classify email
into spam or ham? What activation function should you use in the output layer?
If instead you want to tackle MNIST, how many neurons do you need in the out 
put layer, using what activation function? Answer the same questions for getting
your network to predict housing prices as in Chapter 2.
8. What is backpropagation and how does it work? What is the difference betwe

In [43]:
test_questions = [
    "What is gradient descent and how does it work?",
    "What is the difference between L1 and L2 regularization?",
    "How does dropout prevent overfitting in neural networks?",
    "What is quantum computing?",
    "What does Goodfellow say about optimization in deep learning?"
]

for question in test_questions:
    result = generate_answer(question)
    print(f"\nQUERY: {result['query']}")
    print(f"ANSWER: {result['answer'][:500]}")
    print(f"CITATIONS: {len(result['citations'])} sources")
    print("="*60)
    time.sleep(10)


BUILT PROMPT (SENT TO GEMINI)
SYSTEM RULES:
- Answer ONLY using the context provided below
- Every factual claim must cite its source as:
  [Book Title, Page X]
- If the answer is not in the context say exactly:
  "This information is not available in the loaded books."
- Never invent page numbers or author names
- Answer at graduate student level

CONTEXT:
[1] Book: Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf | Page: 349
7. How many neurons do you need in the output layer if you want to classify email
into spam or ham? What activation function should you use in the output layer?
If instead you want to tackle MNIST, how many neurons do you need in the out 
put layer, using what activation function? Answer the same questions for getting
your network to predict housing prices as in Chapter 2.
8. What is backpropagation and how does it work? What is the difference betwe

In [48]:
# Import pandas so we can build and manage the gold test set DataFrame.
import pandas as pd  # DataFrame construction for evaluation inputs.

# Build the gold test set records as a list of dictionaries.
test_records = [  # Master list containing all 20 required questions.
    # Add factual query 1 with expected retrieval books and answer keywords.
    {"query": "What is the learning rate in gradient descent?", "expected_books": ["Deep Learning", "Burkov"], "expected_keywords": ["learning", "rate", "gradient", "step"], "query_type": "factual", "difficulty": "easy"},
    # Add factual query 2 with expected retrieval books and answer keywords.
    {"query": "What activation function does sigmoid produce?", "expected_books": ["Deep Learning", "Geron"], "expected_keywords": ["sigmoid", "output", "probability", "activation"], "query_type": "factual", "difficulty": "easy"},
    # Add factual query 3 with expected retrieval books and answer keywords.
    {"query": "What does L1 regularization do to model weights?", "expected_books": ["Burkov", "Geron"], "expected_keywords": ["l1", "weights", "sparsity", "regularization"], "query_type": "factual", "difficulty": "easy"},
    # Add factual query 4 with expected retrieval books and answer keywords.
    {"query": "What is a confusion matrix?", "expected_books": ["Geron", "Bruce"], "expected_keywords": ["confusion", "matrix", "true", "false"], "query_type": "factual", "difficulty": "easy"},
    # Add conceptual query 5 with expected retrieval books and answer keywords.
    {"query": "How does backpropagation work?", "expected_books": ["Deep Learning", "Geron"], "expected_keywords": ["gradient", "chain", "error", "weights"], "query_type": "conceptual", "difficulty": "medium"},
    # Add conceptual query 6 with expected retrieval books and answer keywords.
    {"query": "Why does dropout reduce overfitting?", "expected_books": ["Chollet", "Deep Learning"], "expected_keywords": ["dropout", "overfitting", "generalization", "neurons"], "query_type": "conceptual", "difficulty": "medium"},
    # Add conceptual query 7 with expected retrieval books and answer keywords.
    {"query": "What is the curse of dimensionality?", "expected_books": ["Bishop", "Burkov"], "expected_keywords": ["dimensionality", "distance", "sparse", "data"], "query_type": "conceptual", "difficulty": "medium"},
    # Add conceptual query 8 with expected retrieval books and answer keywords.
    {"query": "How do attention mechanisms work in transformers?", "expected_books": ["Jurafsky", "Deep Learning"], "expected_keywords": ["attention", "transformer", "weights", "context"], "query_type": "conceptual", "difficulty": "hard"},
    # Add comparison query 9 with expected retrieval books and answer keywords.
    {"query": "What is the difference between bagging and boosting?", "expected_books": ["Geron", "Burkov"], "expected_keywords": ["bagging", "boosting", "ensemble", "bias"], "query_type": "comparison", "difficulty": "medium"},
    # Add comparison query 10 with expected retrieval books and answer keywords.
    {"query": "How does Adam optimizer differ from SGD?", "expected_books": ["Deep Learning", "Chollet"], "expected_keywords": ["adam", "sgd", "momentum", "adaptive"], "query_type": "comparison", "difficulty": "medium"},
    # Add comparison query 11 with expected retrieval books and answer keywords.
    {"query": "When should I use L1 vs L2 regularization?", "expected_books": ["Burkov", "Geron"], "expected_keywords": ["l1", "l2", "sparsity", "shrinkage"], "query_type": "comparison", "difficulty": "medium"},
    # Add comparison query 12 with expected retrieval books and answer keywords.
    {"query": "What is the difference between RNN and LSTM?", "expected_books": ["Deep Learning", "Jurafsky"], "expected_keywords": ["rnn", "lstm", "memory", "sequence"], "query_type": "comparison", "difficulty": "hard"},
    # Add author-specific query 13 with expected retrieval books and answer keywords.
    {"query": "What does Goodfellow say about vanishing gradients?", "expected_books": ["Deep Learning"], "expected_keywords": ["vanishing", "gradients", "deep", "training"], "query_type": "author_specific", "difficulty": "hard"},
    # Add author-specific query 14 with expected retrieval books and answer keywords.
    {"query": "How does Bishop explain the bias variance tradeoff?", "expected_books": ["Bishop"], "expected_keywords": ["bias", "variance", "tradeoff", "generalization"], "query_type": "author_specific", "difficulty": "hard"},
    # Add author-specific query 15 with expected retrieval books and answer keywords.
    {"query": "What does Chollet say about model capacity?", "expected_books": ["Chollet"], "expected_keywords": ["capacity", "overfitting", "complexity", "model"], "query_type": "author_specific", "difficulty": "medium"},
    # Add author-specific query 16 with expected retrieval books and answer keywords.
    {"query": "What does Burkov say about feature engineering?", "expected_books": ["Burkov"], "expected_keywords": ["feature", "engineering", "representation", "data"], "query_type": "author_specific", "difficulty": "medium"},
    # Add out-of-scope query 17 with expected retrieval books and answer keywords.
    {"query": "What is blockchain technology?", "expected_books": ["Deep Learning"], "expected_keywords": ["outside", "books", "ml", "scope"], "query_type": "out_of_scope", "difficulty": "easy"},
    # Add out-of-scope query 18 with expected retrieval books and answer keywords.
    {"query": "How do I cook pasta?", "expected_books": ["Burkov"], "expected_keywords": ["outside", "books", "ml", "scope"], "query_type": "out_of_scope", "difficulty": "easy"},
    # Add out-of-scope query 19 with expected retrieval books and answer keywords.
    {"query": "What is the history of the Roman Empire?", "expected_books": ["Bishop"], "expected_keywords": ["outside", "books", "ml", "scope"], "query_type": "out_of_scope", "difficulty": "easy"},
    # Add out-of-scope query 20 with expected retrieval books and answer keywords.
    {"query": "How does GPS navigation work?", "expected_books": ["Geron"], "expected_keywords": ["outside", "books", "ml", "scope"], "query_type": "out_of_scope", "difficulty": "easy"}
]  # End of required 20-record gold set.

# Create the requested pandas DataFrame from the test records.
test_df = pd.DataFrame(test_records)  # Final gold test set DataFrame.

# Print a quick shape check so we can verify exactly 20 rows were created.
print(f"Gold test set created with shape: {test_df.shape}")  # Sanity-check output.

# Define retrieval-only evaluation function using hybrid_search and no generation/API calls.
def evaluate_retrieval(test_df):  # Evaluate retrieval quality on non-out-of-scope queries only.
    # Filter to the 16 in-scope queries by excluding out_of_scope query type.
    eval_df = test_df[test_df["query_type"] != "out_of_scope"].copy()  # Keep only in-scope rows.
    # Initialize list to collect per-query retrieval evaluation records.
    evaluation_rows = []  # Stores row-level hit and rank diagnostics.
    # Loop through each evaluation query row.
    for _, row in eval_df.iterrows():  # Iterate through in-scope test queries.
        # Extract the query string from the DataFrame row.
        query = row["query"]  # Current question text.
        # Extract expected partial book-name matches for this query.
        expected_books = row["expected_books"]  # List of expected book partial names.
        # Run hybrid retrieval for top-5 chunks exactly as requested.
        retrieved_chunks = hybrid_search(query, top_k=5)  # Top-5 retrieval results.
        # Build top-5 retrieved book title list from returned chunks.
        retrieved_book_titles = [str(chunk.get("book_title", "")) for chunk in retrieved_chunks]  # Retrieved titles.
        # Normalize expected book patterns to lowercase for case-insensitive matching.
        expected_books_lower = [book.lower() for book in expected_books]  # Lowercased expected partial names.
        # Normalize retrieved titles to lowercase for case-insensitive matching.
        retrieved_book_titles_lower = [title.lower() for title in retrieved_book_titles]  # Lowercased retrieved titles.
        # Initialize hit flag as False before scanning top-5 titles.
        hit = False  # Whether any expected book appears in retrieved top-5.
        # Initialize rank as 999 to indicate not-found default.
        first_hit_rank = 999  # Rank placeholder when no correct title is found.
        # Scan retrieved titles in rank order to find earliest expected match.
        for idx, retrieved_title in enumerate(retrieved_book_titles_lower, start=1):  # 1-based rank scan.
            # Check if any expected partial name appears inside this retrieved title.
            if any(expected_book in retrieved_title for expected_book in expected_books_lower):  # Partial-match check.
                # Mark this query as a retrieval hit.
                hit = True  # Hit achieved.
                # Record first matching rank position.
                first_hit_rank = idx  # Earliest correct rank.
                # Stop scan once first match is found.
                break  # Exit rank loop on first hit.
        # Append full per-query diagnostic record to evaluation rows.
        evaluation_rows.append({  # One output row per evaluated query.
            "query": query,  # Original query text.
            "expected_books": expected_books,  # Expected book partial names.
            "retrieved_book_titles": retrieved_book_titles,  # Top-5 retrieved book titles.
            "hit": hit,  # Boolean hit indicator.
            "rank": first_hit_rank  # First hit rank or 999.
        })  # End one evaluation row.
    # Convert evaluation records to a DataFrame for metric aggregation and debugging.
    evaluation_df = pd.DataFrame(evaluation_rows)  # Structured retrieval evaluation results.
    # Count total evaluated queries (should be 16).
    total_queries = len(evaluation_df)  # Number of in-scope queries evaluated.
    # Count retrieval hits where at least one expected book appears in top-5.
    total_hits = int(evaluation_df["hit"].sum())  # Total hit count.
    # Compute Recall@5 as hits divided by total non-out-of-scope queries.
    recall_at_5 = (total_hits / total_queries) if total_queries > 0 else 0.0  # Recall@5 metric.
    # Compute reciprocal rank per query using 1/rank with 999 for misses.
    reciprocal_ranks = [1.0 / rank for rank in evaluation_df["rank"].tolist()]  # RR list including misses.
    # Compute MRR as the mean reciprocal rank across all evaluated queries.
    mrr = (sum(reciprocal_ranks) / total_queries) if total_queries > 0 else 0.0  # Mean Reciprocal Rank metric.
    # Build passed subset where hit is True.
    passed_df = evaluation_df[evaluation_df["hit"] == True]  # Successful retrieval cases.
    # Build failed subset where hit is False.
    failed_df = evaluation_df[evaluation_df["hit"] == False]  # Failed retrieval cases.
    # Print exact report header line.
    print("RETRIEVAL EVALUATION REPORT")  # Required report title.
    # Print exact report underline line.
    print("============================")  # Required separator.
    # Print total evaluated queries line.
    print(f"Total queries evaluated: {total_queries}")  # Query count line.
    # Print Recall@5 line with 2 decimals as requested.
    print(f"Recall@5: {recall_at_5:.2f}")  # Recall metric line.
    # Print MRR line with 2 decimals as requested.
    print(f"MRR: {mrr:.2f}")  # MRR metric line.
    # Print a blank line for section readability.
    print()  # Spacer line.
    # Print passed section header with hit count.
    print(f"PASSED ({len(passed_df)} queries):")  # Passed section heading.
    # Loop through passed rows and print requested success format.
    for _, row in passed_df.iterrows():  # Iterate successful rows.
        # Find which expected book matched first for clearer reporting.
        matched_book = None  # Placeholder for matched expected pattern.
        # Iterate expected patterns in listed order.
        for expected_book in row["expected_books"]:  # Scan expected books.
            # Check whether expected partial appears in any retrieved title.
            if any(expected_book.lower() in title.lower() for title in row["retrieved_book_titles"]):  # Match test.
                # Store first matching expected book label.
                matched_book = expected_book  # Matched expected identifier.
                # Stop once first expected match is found.
                break  # Exit expected book loop.
        # Fallback label if no explicit expected substring is identified.
        if matched_book is None:  # Handle rare edge case.
            matched_book = "expected_book"  # Safe fallback label.
        # Print one passed line in the requested style.
        print(f"  ✅ \"{row['query']}\" → found {matched_book} rank {row['rank']}")  # Passed query line.
    # Print blank line before failed section.
    print()  # Spacer line.
    # Print failed section header with failure count.
    print(f"FAILED ({len(failed_df)} queries):")  # Failed section heading.
    # Loop through failed rows and print requested failure format.
    for _, row in failed_df.iterrows():  # Iterate failed rows.
        # Join expected book patterns as comma-separated string.
        expected_books_text = ", ".join(row["expected_books"])  # Expected books text.
        # Print first line of failed entry with query and expected books.
        print(f"  ❌ \"{row['query']}\" → expected {expected_books_text},")  # Failed query line 1.
        # Print second line with retrieved top-5 book titles list.
        print(f"     got: {row['retrieved_book_titles']}")  # Failed query line 2.
    # Return detailed evaluation DataFrame and summary metrics for downstream use.
    return evaluation_df, recall_at_5, mrr  # Structured return payload.

# Run retrieval-only evaluation now as requested.
retrieval_eval_df, recall_at_5, mrr = evaluate_retrieval(test_df)  # Execute retrieval evaluation.

Gold test set created with shape: (20, 5)
RETRIEVAL EVALUATION REPORT
Total queries evaluated: 16
Recall@5: 0.75
MRR: 0.50

PASSED (12 queries):
  ✅ "What is the learning rate in gradient descent?" → found Deep Learning rank 1
  ✅ "What activation function does sigmoid produce?" → found Deep Learning rank 2
  ✅ "What does L1 regularization do to model weights?" → found Burkov rank 2
  ✅ "What is a confusion matrix?" → found Bruce rank 1
  ✅ "How does backpropagation work?" → found Deep Learning rank 2
  ✅ "Why does dropout reduce overfitting?" → found Deep Learning rank 1
  ✅ "How do attention mechanisms work in transformers?" → found Deep Learning rank 4
  ✅ "When should I use L1 vs L2 regularization?" → found Burkov rank 1
  ✅ "What is the difference between RNN and LSTM?" → found Deep Learning rank 5
  ✅ "What does Goodfellow say about vanishing gradients?" → found Deep Learning rank 2
  ✅ "How does Bishop explain the bias variance tradeoff?" → found Bishop rank 1
  ✅ "What does Bur